# Notebook 06 — System Pressure Trend Analysis & Executive Insights

## Step 6A — Notebook Setup, Library Imports & Final Dataset Loading

### Objective
This step initializes Notebook 06 by importing required Python libraries, configuring display settings, defining project paths, and loading the final pressure and stress classification dataset exported from Notebook 05.

### Input Dataset
The input dataset for this notebook is:

`data/processed/final_pressure_stress_classification_dataset.csv`

This dataset contains the final pressure and stress classification outputs created in Notebook 05, including system load, intake, backlog severity, volatility, pressure spike, daily strain, prolonged pressure, and final pressure classification indicators.

### Step 6A Tasks
- Import required Python libraries
- Configure notebook display settings
- Define input and output paths
- Load the final pressure and stress classification dataset
- Validate dataset availability
- Check dataset shape, columns, missing values, date parsing, and chronological order
- Create a safe working copy for Notebook 06

### Expected Output
A clean and validated Notebook 06 working dataset ready for trend analysis, pressure pattern analysis, executive summaries, and dashboard-ready exports.

In [1]:
# ============================================================
# Step 6A - Notebook 06 Setup and Creation,
# Library Imports & Final Dataset Loading
# ============================================================

# ------------------------------
# 1. Import required libraries
# ------------------------------

import os
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")


# ------------------------------
# 2. Configure notebook display
# ------------------------------

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)
pd.set_option("display.width", 120)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")


# ------------------------------
# 3. Define project paths
# ------------------------------

input_path = Path("data/processed/final_pressure_stress_classification_dataset.csv")
output_dir = Path("data/processed")
output_dir.mkdir(parents=True, exist_ok=True)


# ------------------------------
# 4. Validate input file existence
# ------------------------------

print("Step 6A - Notebook 06 Setup and Final Dataset Loading")
print("=" * 75)

if not input_path.exists():
    raise FileNotFoundError(
        f"Input dataset not found at: {input_path}\n"
        "Please confirm that Notebook 05 exported the final dataset correctly."
    )

print("Input dataset found successfully.")
print(f"Input path: {input_path}")


# ------------------------------
# 5. Load final pressure dataset
# ------------------------------

df_pressure_final = pd.read_csv(input_path)

print("\nFinal pressure and stress classification dataset loaded successfully.")
print(f"Dataset shape: {df_pressure_final.shape[0]} rows x {df_pressure_final.shape[1]} columns")


# ------------------------------
# 6. Standardize and validate Date column
# ------------------------------

if "Date" not in df_pressure_final.columns:
    raise KeyError(
        "Required column 'Date' is missing from the final pressure dataset."
    )

df_pressure_final["Date"] = pd.to_datetime(df_pressure_final["Date"], errors="coerce")

date_parse_errors = df_pressure_final["Date"].isna().sum()

if date_parse_errors > 0:
    raise ValueError(
        f"Date parsing issue found. Number of invalid Date values: {date_parse_errors}"
    )

df_pressure_final = df_pressure_final.sort_values("Date").reset_index(drop=True)

chronological_order_check = df_pressure_final["Date"].is_monotonic_increasing
duplicate_date_count = df_pressure_final["Date"].duplicated().sum()


# ------------------------------
# 7. Create safe working copy
# ------------------------------

df_notebook_06 = df_pressure_final.copy()


# ------------------------------
# 8. Required core column validation
# ------------------------------

required_base_columns = [
    "Date",
    "total_system_load",
    "net_daily_intake",
    "care_load_growth_rate_pct",
    "backlog_severity_score",
]

missing_base_columns = [
    col for col in required_base_columns
    if col not in df_notebook_06.columns
]

if missing_base_columns:
    raise KeyError(
        f"Missing required base columns for Notebook 06: {missing_base_columns}"
    )


# ------------------------------
# 9. Important pressure classification column validation
# ------------------------------

important_pressure_columns = [
    "volatility_pressure_score",
    "pressure_spike_score",
    "daily_strain_score",
    "final_pressure_stress_score",
    "final_pressure_stress_level",
    "operational_response_category",
    "primary_pressure_driver",
]

available_pressure_columns = [
    col for col in important_pressure_columns
    if col in df_notebook_06.columns
]

missing_pressure_columns = [
    col for col in important_pressure_columns
    if col not in df_notebook_06.columns
]


# ------------------------------
# 10. Dataset quality checks
# ------------------------------

total_missing_values = int(df_notebook_06.isna().sum().sum())
total_duplicate_rows = int(df_notebook_06.duplicated().sum())

date_min = df_notebook_06["Date"].min()
date_max = df_notebook_06["Date"].max()

numeric_columns = df_notebook_06.select_dtypes(include=[np.number]).columns.tolist()
categorical_columns = df_notebook_06.select_dtypes(include=["object", "category"]).columns.tolist()


# ------------------------------
# 11. Create Step 6A validation summary
# ------------------------------

step_6a_validation_summary = pd.DataFrame({
    "Step 6A Check": [
        "Input Dataset Path",
        "Input File Exists",
        "Rows Loaded",
        "Columns Loaded",
        "Date Parse Errors",
        "Duplicate Date Count",
        "Chronological Order",
        "Total Missing Values",
        "Duplicate Full Rows",
        "Required Base Columns Missing",
        "Important Pressure Columns Available",
        "Important Pressure Columns Missing",
        "Numeric Columns Count",
        "Categorical Columns Count",
        "Minimum Date",
        "Maximum Date",
    ],
    "Result": [
        str(input_path),
        input_path.exists(),
        df_notebook_06.shape[0],
        df_notebook_06.shape[1],
        date_parse_errors,
        duplicate_date_count,
        chronological_order_check,
        total_missing_values,
        total_duplicate_rows,
        len(missing_base_columns),
        len(available_pressure_columns),
        len(missing_pressure_columns),
        len(numeric_columns),
        len(categorical_columns),
        date_min,
        date_max,
    ]
})


# ------------------------------
# 12. Display validation summary
# ------------------------------

print("\nStep 6A validation summary:")
display(step_6a_validation_summary)


# ------------------------------
# 13. Display available important pressure columns
# ------------------------------

pressure_column_status = pd.DataFrame({
    "Pressure Column": important_pressure_columns,
    "Status": [
        "Available" if col in df_notebook_06.columns else "Missing"
        for col in important_pressure_columns
    ]
})

print("\nImportant pressure classification column status:")
display(pressure_column_status)


# ------------------------------
# 14. Display dataset preview
# ------------------------------

preview_columns = [
    "Date",
    "total_system_load",
    "net_daily_intake",
    "care_load_growth_rate_pct",
    "backlog_severity_score",
    "volatility_pressure_score",
    "pressure_spike_score",
    "daily_strain_score",
    "final_pressure_stress_score",
    "final_pressure_stress_level",
    "operational_response_category",
    "primary_pressure_driver",
]

preview_columns = [
    col for col in preview_columns
    if col in df_notebook_06.columns
]

print("\nNotebook 06 working dataset preview:")
display(df_notebook_06[preview_columns].head(10))

print("\nNotebook 06 working dataset tail:")
display(df_notebook_06[preview_columns].tail(10))


# ------------------------------
# 15. Completion message
# ------------------------------

print("\nStep 6A completed successfully.")
print("Notebook 06 setup is complete and the final pressure dataset is ready for Step 6B.")

Step 6A - Notebook 06 Setup and Final Dataset Loading
Input dataset found successfully.
Input path: data\processed\final_pressure_stress_classification_dataset.csv

Final pressure and stress classification dataset loaded successfully.
Dataset shape: 720 rows x 107 columns

Step 6A validation summary:


,Step 6A Check,Result
0,Input Dataset Path,data\processed\final_pressure_stress_classific...
1,Input File Exists,True
2,Rows Loaded,720
3,Columns Loaded,107
4,Date Parse Errors,0
5,Duplicate Date Count,0
6,Chronological Order,True
7,Total Missing Values,331
8,Duplicate Full Rows,0
9,Required Base Columns Missing,0



Important pressure classification column status:


,Pressure Column,Status
0,volatility_pressure_score,Available
1,pressure_spike_score,Available
2,daily_strain_score,Available
3,final_pressure_stress_score,Available
4,final_pressure_stress_level,Available
5,operational_response_category,Available
6,primary_pressure_driver,Available



Notebook 06 working dataset preview:


,Date,total_system_load,net_daily_intake,care_load_growth_rate_pct,backlog_severity_score,volatility_pressure_score,pressure_spike_score,daily_strain_score,final_pressure_stress_score,final_pressure_stress_level,operational_response_category,primary_pressure_driver
0,2023-01-12,6619,-402,0.00,0,0,0,0,0.99,Normal Operating Pressure,Routine Monitoring,No Major Pressure Driver
1,2023-01-22,7171,-188,8.34,0,5,6,7,6.07,Critical System Pressure,Immediate Escalation Required,Sudden Spike Pressure
2,2023-01-23,7330,-142,2.22,0,5,2,7,4.70,Critical System Pressure,Immediate Escalation Required,Prolonged Window Pressure
3,2023-01-24,7475,-128,1.98,0,5,0,4,3.49,High System Stress,Priority Monitoring Required,Prolonged Window Pressure
4,2023-01-25,7560,-139,1.14,0,5,0,4,3.48,High System Stress,Priority Monitoring Required,Prolonged Window Pressure
5,2023-01-29,7517,-292,-0.57,0,5,2,7,4.84,Critical System Pressure,Immediate Escalation Required,Prolonged Window Pressure
6,2023-01-30,7797,-167,3.72,0,5,3,7,5.11,Critical System Pressure,Immediate Escalation Required,Prolonged Window Pressure
7,2023-01-31,7839,-122,0.54,0,5,0,3,3.24,Moderate System Stress,Operational Watch Required,Prolonged Window Pressure
8,2023-02-01,7935,-204,1.22,0,4,0,3,3.08,Moderate System Stress,Operational Watch Required,Prolonged Window Pressure
9,2023-02-02,7892,-275,-0.54,0,4,0,4,3.36,Moderate System Stress,Operational Watch Required,Prolonged Window Pressure



Notebook 06 working dataset tail:


,Date,total_system_load,net_daily_intake,care_load_growth_rate_pct,backlog_severity_score,volatility_pressure_score,pressure_spike_score,daily_strain_score,final_pressure_stress_score,final_pressure_stress_level,operational_response_category,primary_pressure_driver
710,2025-12-08,2467,5,0.57,1,0,0,3,0.91,Moderate System Stress,Operational Watch Required,Backlog Severity Pressure
711,2025-12-09,2473,-1,0.24,0,0,0,2,0.50,Early Pressure Watch,Early Warning Review,Daily Strain Pressure
712,2025-12-10,2493,-4,0.81,0,0,0,2,0.53,Early Pressure Watch,Early Warning Review,Daily Strain Pressure
713,2025-12-11,2484,-1,-0.36,0,0,0,2,0.51,Early Pressure Watch,Early Warning Review,Daily Strain Pressure
714,2025-12-14,2497,-4,0.52,0,0,0,2,0.52,Early Pressure Watch,Early Warning Review,Daily Strain Pressure
715,2025-12-15,2512,2,0.60,1,0,1,7,2.35,High System Stress,Priority Monitoring Required,Prolonged Window Pressure
716,2025-12-16,2522,6,0.40,1,0,0,3,1.20,Moderate System Stress,Operational Watch Required,Prolonged Window Pressure
717,2025-12-17,2512,1,-0.40,2,1,1,9,3.25,Critical System Pressure,Immediate Escalation Required,Prolonged Window Pressure
718,2025-12-18,2522,-10,0.40,0,0,0,2,0.52,Early Pressure Watch,Early Warning Review,Daily Strain Pressure
719,2025-12-21,2502,-3,-0.79,0,0,0,0,0.09,Normal Operating Pressure,Routine Monitoring,No Major Pressure Driver



Step 6A completed successfully.
Notebook 06 setup is complete and the final pressure dataset is ready for Step 6B.


## Step 6B — Create Time-Based Pressure Trend Aggregations

In this step, the final pressure and stress classification dataset is aggregated into time-based trend views.

The objective of this step is to create daily, weekly, monthly, quarterly, and yearly pressure trend summaries so that the system pressure movement can be analyzed across different reporting periods.

This step uses the final pressure stress score, pressure classification, operational response category, and key system load indicators from the final exported dataset created in Notebook 05.

The output of this step will be used in the next steps for executive trend analysis, peak pressure detection, long-term movement patterns, and time-based operational insight generation.

In [2]:
# ============================================================
# Step 6B - Create Time-Based Pressure Trend Aggregations
# ============================================================

import pandas as pd
import numpy as np

print("Step 6B - Create Time-Based Pressure Trend Aggregations")
print("=" * 75)

# ------------------------------------------------------------
# 1. Validate Step 6A dataset availability
# ------------------------------------------------------------

if "df_notebook_06" not in globals():
    raise NameError(
        "df_notebook_06 not found. Please run Step 6A before running Step 6B."
    )

df_trend_base = df_notebook_06.copy()

print("Trend base working copy created successfully.")
print(f"Input dataset shape: {df_trend_base.shape[0]} rows x {df_trend_base.shape[1]} columns")

# ------------------------------------------------------------
# 2. Required column validation
# ------------------------------------------------------------

required_step_6b_columns = [
    "Date",
    "total_system_load",
    "net_daily_intake",
    "care_load_growth_rate_pct",
    "backlog_severity_score",
    "volatility_pressure_score",
    "pressure_spike_score",
    "daily_strain_score",
    "final_pressure_stress_score",
    "final_pressure_stress_level",
    "operational_response_category",
    "primary_pressure_driver"
]

missing_step_6b_columns = [
    col for col in required_step_6b_columns if col not in df_trend_base.columns
]

if missing_step_6b_columns:
    raise ValueError(
        f"Missing required Step 6B columns: {missing_step_6b_columns}"
    )

print(f"Missing Step 6B required columns: {len(missing_step_6b_columns)}")

# ------------------------------------------------------------
# 3. Date preparation and chronological validation
# ------------------------------------------------------------

df_trend_base["Date"] = pd.to_datetime(df_trend_base["Date"], errors="coerce")

date_parse_errors_6b = int(df_trend_base["Date"].isna().sum())

if date_parse_errors_6b > 0:
    raise ValueError(
        f"Date parsing issue found in Step 6B. Invalid Date rows: {date_parse_errors_6b}"
    )

df_trend_base = df_trend_base.sort_values("Date").reset_index(drop=True)

duplicate_dates_6b = int(df_trend_base["Date"].duplicated().sum())
chronological_order_6b = bool(df_trend_base["Date"].is_monotonic_increasing)

print(f"Date parse errors: {date_parse_errors_6b}")
print(f"Duplicate dates: {duplicate_dates_6b}")
print(f"Dataset chronological order: {chronological_order_6b}")

# ------------------------------------------------------------
# 4. Create time period columns
# ------------------------------------------------------------

df_trend_base["year"] = df_trend_base["Date"].dt.year
df_trend_base["quarter"] = df_trend_base["Date"].dt.to_period("Q").astype(str)
df_trend_base["month"] = df_trend_base["Date"].dt.to_period("M").astype(str)
df_trend_base["week"] = df_trend_base["Date"].dt.to_period("W").astype(str)

df_trend_base["month_start_date"] = df_trend_base["Date"].values.astype("datetime64[M]")
df_trend_base["quarter_start_date"] = df_trend_base["Date"].dt.to_period("Q").dt.start_time
df_trend_base["week_start_date"] = df_trend_base["Date"].dt.to_period("W").dt.start_time

print("Time period columns created successfully.")

# ------------------------------------------------------------
# 5. Helper function for pressure trend aggregation
# ------------------------------------------------------------

def create_pressure_trend_summary(data, group_col, period_label):
    """
    Creates time-based pressure trend aggregation for one selected period.
    """

    trend_summary = (
        data
        .groupby(group_col, as_index=False)
        .agg(
            reporting_days=("Date", "count"),
            period_start_date=("Date", "min"),
            period_end_date=("Date", "max"),
            avg_system_load=("total_system_load", "mean"),
            max_system_load=("total_system_load", "max"),
            min_system_load=("total_system_load", "min"),
            avg_net_daily_intake=("net_daily_intake", "mean"),
            total_net_daily_intake=("net_daily_intake", "sum"),
            avg_care_load_growth_pct=("care_load_growth_rate_pct", "mean"),
            avg_backlog_severity_score=("backlog_severity_score", "mean"),
            max_backlog_severity_score=("backlog_severity_score", "max"),
            avg_volatility_pressure_score=("volatility_pressure_score", "mean"),
            max_volatility_pressure_score=("volatility_pressure_score", "max"),
            avg_pressure_spike_score=("pressure_spike_score", "mean"),
            max_pressure_spike_score=("pressure_spike_score", "max"),
            avg_daily_strain_score=("daily_strain_score", "mean"),
            max_daily_strain_score=("daily_strain_score", "max"),
            avg_final_pressure_stress_score=("final_pressure_stress_score", "mean"),
            max_final_pressure_stress_score=("final_pressure_stress_score", "max"),
            normal_pressure_days=(
                "final_pressure_stress_level",
                lambda x: (x == "Normal Operating Pressure").sum()
            ),
            early_watch_days=(
                "final_pressure_stress_level",
                lambda x: (x == "Early Pressure Watch").sum()
            ),
            moderate_stress_days=(
                "final_pressure_stress_level",
                lambda x: (x == "Moderate System Stress").sum()
            ),
            high_stress_days=(
                "final_pressure_stress_level",
                lambda x: (x == "High System Stress").sum()
            ),
            critical_pressure_days=(
                "final_pressure_stress_level",
                lambda x: (x == "Critical System Pressure").sum()
            ),
            most_common_pressure_level=(
                "final_pressure_stress_level",
                lambda x: x.mode().iloc[0] if not x.mode().empty else "Not Available"
            ),
            most_common_response_category=(
                "operational_response_category",
                lambda x: x.mode().iloc[0] if not x.mode().empty else "Not Available"
            ),
            most_common_pressure_driver=(
                "primary_pressure_driver",
                lambda x: x.mode().iloc[0] if not x.mode().empty else "Not Available"
            )
        )
    )

    trend_summary["trend_period_type"] = period_label

    trend_summary["stress_day_count"] = (
        trend_summary["moderate_stress_days"]
        + trend_summary["high_stress_days"]
        + trend_summary["critical_pressure_days"]
    )

    trend_summary["stress_day_rate_pct"] = np.where(
        trend_summary["reporting_days"] > 0,
        (trend_summary["stress_day_count"] / trend_summary["reporting_days"]) * 100,
        0
    )

    trend_summary["critical_pressure_rate_pct"] = np.where(
        trend_summary["reporting_days"] > 0,
        (trend_summary["critical_pressure_days"] / trend_summary["reporting_days"]) * 100,
        0
    )

    trend_summary["high_or_critical_pressure_days"] = (
        trend_summary["high_stress_days"] + trend_summary["critical_pressure_days"]
    )

    trend_summary["high_or_critical_pressure_rate_pct"] = np.where(
        trend_summary["reporting_days"] > 0,
        (
            trend_summary["high_or_critical_pressure_days"]
            / trend_summary["reporting_days"]
        ) * 100,
        0
    )

    round_columns = [
        "avg_system_load",
        "avg_net_daily_intake",
        "avg_care_load_growth_pct",
        "avg_backlog_severity_score",
        "avg_volatility_pressure_score",
        "avg_pressure_spike_score",
        "avg_daily_strain_score",
        "avg_final_pressure_stress_score",
        "stress_day_rate_pct",
        "critical_pressure_rate_pct",
        "high_or_critical_pressure_rate_pct"
    ]

    for col in round_columns:
        if col in trend_summary.columns:
            trend_summary[col] = trend_summary[col].round(2)

    return trend_summary

# ------------------------------------------------------------
# 6. Create daily, weekly, monthly, quarterly, and yearly trends
# ------------------------------------------------------------

df_daily_pressure_trend = create_pressure_trend_summary(
    data=df_trend_base,
    group_col="Date",
    period_label="Daily"
)

df_weekly_pressure_trend = create_pressure_trend_summary(
    data=df_trend_base,
    group_col="week",
    period_label="Weekly"
)

df_monthly_pressure_trend = create_pressure_trend_summary(
    data=df_trend_base,
    group_col="month",
    period_label="Monthly"
)

df_quarterly_pressure_trend = create_pressure_trend_summary(
    data=df_trend_base,
    group_col="quarter",
    period_label="Quarterly"
)

df_yearly_pressure_trend = create_pressure_trend_summary(
    data=df_trend_base,
    group_col="year",
    period_label="Yearly"
)

print("Time-based pressure trend aggregations created successfully.")

# ------------------------------------------------------------
# 7. Create combined trend dataset
# ------------------------------------------------------------

df_daily_pressure_trend = df_daily_pressure_trend.rename(columns={"Date": "trend_period"})
df_weekly_pressure_trend = df_weekly_pressure_trend.rename(columns={"week": "trend_period"})
df_monthly_pressure_trend = df_monthly_pressure_trend.rename(columns={"month": "trend_period"})
df_quarterly_pressure_trend = df_quarterly_pressure_trend.rename(columns={"quarter": "trend_period"})
df_yearly_pressure_trend = df_yearly_pressure_trend.rename(columns={"year": "trend_period"})

df_combined_pressure_trend = pd.concat(
    [
        df_daily_pressure_trend,
        df_weekly_pressure_trend,
        df_monthly_pressure_trend,
        df_quarterly_pressure_trend,
        df_yearly_pressure_trend
    ],
    ignore_index=True
)

df_combined_pressure_trend["trend_period"] = df_combined_pressure_trend["trend_period"].astype(str)

print(f"Combined trend dataset shape: {df_combined_pressure_trend.shape[0]} rows x {df_combined_pressure_trend.shape[1]} columns")

# ------------------------------------------------------------
# 8. Step 6B validation summary
# ------------------------------------------------------------

step_6b_created_datasets = {
    "Daily Trend Rows": len(df_daily_pressure_trend),
    "Weekly Trend Rows": len(df_weekly_pressure_trend),
    "Monthly Trend Rows": len(df_monthly_pressure_trend),
    "Quarterly Trend Rows": len(df_quarterly_pressure_trend),
    "Yearly Trend Rows": len(df_yearly_pressure_trend),
    "Combined Trend Rows": len(df_combined_pressure_trend)
}

step_6b_missing_values = int(
    df_combined_pressure_trend[
        [
            "trend_period",
            "trend_period_type",
            "reporting_days",
            "avg_final_pressure_stress_score",
            "max_final_pressure_stress_score",
            "most_common_pressure_level",
            "most_common_response_category",
            "most_common_pressure_driver"
        ]
    ].isna().sum().sum()
)

step_6b_validation_summary = pd.DataFrame({
    "Step 6B Check": [
        "Input Dataset Source",
        "Rows Checked",
        "Columns Before Step 6B",
        "Daily Trend Rows",
        "Weekly Trend Rows",
        "Monthly Trend Rows",
        "Quarterly Trend Rows",
        "Yearly Trend Rows",
        "Combined Trend Rows",
        "Missing Required Input Columns",
        "Date Parse Errors",
        "Duplicate Dates",
        "Chronological Order",
        "Missing Values in Core Trend Columns"
    ],
    "Result": [
        "df_notebook_06",
        df_trend_base.shape[0],
        df_notebook_06.shape[1],
        step_6b_created_datasets["Daily Trend Rows"],
        step_6b_created_datasets["Weekly Trend Rows"],
        step_6b_created_datasets["Monthly Trend Rows"],
        step_6b_created_datasets["Quarterly Trend Rows"],
        step_6b_created_datasets["Yearly Trend Rows"],
        step_6b_created_datasets["Combined Trend Rows"],
        len(missing_step_6b_columns),
        date_parse_errors_6b,
        duplicate_dates_6b,
        chronological_order_6b,
        step_6b_missing_values
    ]
})

print("\nStep 6B validation summary:")
display(step_6b_validation_summary)

# ------------------------------------------------------------
# 9. Display trend distribution summaries
# ------------------------------------------------------------

trend_period_distribution = (
    df_combined_pressure_trend["trend_period_type"]
    .value_counts()
    .reset_index()
)

trend_period_distribution.columns = [
    "Trend Period Type",
    "Record Count"
]

print("\nTrend period distribution:")
display(trend_period_distribution)

print("\nMonthly pressure trend preview:")
display(df_monthly_pressure_trend.head(10))

print("\nQuarterly pressure trend preview:")
display(df_quarterly_pressure_trend.head(10))

print("\nCombined pressure trend preview:")
display(df_combined_pressure_trend.head(10))

# ------------------------------------------------------------
# 10. Completion message
# ------------------------------------------------------------

print("\nStep 6B completed successfully.")
print("Time-based pressure trend aggregations are ready for Step 6C.")

Step 6B - Create Time-Based Pressure Trend Aggregations
Trend base working copy created successfully.
Input dataset shape: 720 rows x 107 columns
Missing Step 6B required columns: 0
Date parse errors: 0
Duplicate dates: 0
Dataset chronological order: True
Time period columns created successfully.
Time-based pressure trend aggregations created successfully.
Combined trend dataset shape: 925 rows x 34 columns

Step 6B validation summary:


,Step 6B Check,Result
0,Input Dataset Source,df_notebook_06
1,Rows Checked,720
2,Columns Before Step 6B,107
3,Daily Trend Rows,720
4,Weekly Trend Rows,154
5,Monthly Trend Rows,36
6,Quarterly Trend Rows,12
7,Yearly Trend Rows,3
8,Combined Trend Rows,925
9,Missing Required Input Columns,0



Trend period distribution:


,Trend Period Type,Record Count
0,Daily,720
1,Weekly,154
2,Monthly,36
3,Quarterly,12
4,Yearly,3



Monthly pressure trend preview:


...


Quarterly pressure trend preview:


,trend_period,reporting_days,period_start_date,period_end_date,avg_system_load,max_system_load,min_system_load,avg_net_daily_intake,total_net_daily_intake,avg_care_load_growth_pct,avg_backlog_severity_score,max_backlog_severity_score,avg_volatility_pressure_score,max_volatility_pressure_score,avg_pressure_spike_score,max_pressure_spike_score,avg_daily_strain_score,max_daily_strain_score,avg_final_pressure_stress_score,max_final_pressure_stress_score,normal_pressure_days,early_watch_days,moderate_stress_days,high_stress_days,critical_pressure_days,most_common_pressure_level,most_common_response_category,most_common_pressure_driver,trend_period_type,stress_day_count,stress_day_rate_pct,critical_pressure_rate_pct,high_or_critical_pressure_days,high_or_critical_pressure_rate_pct
0,2023Q1,46,2023-01-12,2023-03-30,"7,963.09",8453,6619,-132.65,-6102,0.55,0.04,1,2.30,5,0.98,6,3.87,8,2.81,6.07,2,15,11,11,7,Early Pressure Watch,Early Warning Review,Prolonged Window Pressure,Quarterly,29,63.04,15.22,18,39.13
1,2023Q2,63,2023-04-02,2023-06-29,"7,626.70",9084,5730,-94.41,-5948,-0.43,0.06,1,1.83,6,0.86,5,2.63,8,2.18,5.67,13,25,8,13,4,Early Pressure Watch,Early Warning Review,Volatility Pressure,Quarterly,25,39.68,6.35,17,26.98
2,2023Q3,60,2023-07-02,2023-09-28,"9,151.63",11382,6188,-99.47,-5968,0.98,0.10,1,1.98,5,0.73,6,4.10,9,2.80,5.48,1,21,13,15,10,Early Pressure Watch,Early Warning Review,Prolonged Window Pressure,Quarterly,38,63.33,16.67,25,41.67
3,2023Q4,61,2023-10-01,2023-12-28,"10,472.51",11762,9000,-198.39,-12102,0.04,0.00,0,1.87,5,0.85,5,3.41,7,2.80,5.06,5,19,12,18,7,Early Pressure Watch,Early Warning Review,Daily Strain Pressure,Quarterly,37,60.66,11.48,25,40.98
4,2024Q1,63,2024-01-01,2024-03-31,"8,834.81",10994,7511,-48.51,-3056,-0.63,0.46,2,2.29,6,1.05,5,3.89,10,2.95,5.41,11,7,18,14,13,Moderate System Stress,Operational Watch Required,Prolonged Window Pressure,Quarterly,45,71.43,20.63,27,42.86
5,2024Q2,64,2024-04-01,2024-06-30,"7,616.83",8067,6875,23.95,1533,-0.07,1.22,3,1.08,3,0.75,5,4.34,9,2.55,4.62,7,8,31,8,10,Moderate System Stress,Operational Watch Required,Prolonged Window Pressure,Quarterly,49,76.56,15.62,18,28.12
6,2024Q3,62,2024-07-01,2024-09-30,"6,408.84",7058,6029,8.84,548,-0.19,0.97,3,0.39,2,0.56,3,3.76,9,1.93,4.16,16,8,21,10,7,Moderate System Stress,Operational Watch Required,Prolonged Window Pressure,Quarterly,38,61.29,11.29,17,27.42
7,2024Q4,62,2024-10-01,2024-12-30,"6,384.97",7012,5947,29.65,1838,0.10,1.34,3,0.45,1,0.45,2,4.73,10,2.28,4.22,5,9,28,9,11,Moderate System Stress,Operational Watch Required,Prolonged Window Pressure,Quarterly,48,77.42,17.74,20,32.26
8,2025Q1,63,2025-01-01,2025-03-31,"3,383.03",6471,2125,-39.97,-2518,-1.68,0.38,3,1.94,4,0.65,4,2.52,9,1.70,4.21,16,23,13,8,3,Early Pressure Watch,Early Warning Review,Volatility Pressure,Quarterly,24,38.10,4.76,11,17.46
9,2025Q2,58,2025-04-01,2025-06-30,"2,404.57",2552,2227,-1.19,-69,0.16,0.55,2,0.10,1,0.24,1,3.21,9,1.02,3.27,11,21,13,11,2,Early Pressure Watch,Early Warning Review,Daily Strain Pressure,Quarterly,26,44.83,3.45,13,22.41



Combined pressure trend preview:


,trend_period,reporting_days,period_start_date,period_end_date,avg_system_load,max_system_load,min_system_load,avg_net_daily_intake,total_net_daily_intake,avg_care_load_growth_pct,avg_backlog_severity_score,max_backlog_severity_score,avg_volatility_pressure_score,max_volatility_pressure_score,avg_pressure_spike_score,max_pressure_spike_score,avg_daily_strain_score,max_daily_strain_score,avg_final_pressure_stress_score,max_final_pressure_stress_score,normal_pressure_days,early_watch_days,moderate_stress_days,high_stress_days,critical_pressure_days,most_common_pressure_level,most_common_response_category,most_common_pressure_driver,trend_period_type,stress_day_count,stress_day_rate_pct,critical_pressure_rate_pct,high_or_critical_pressure_days,high_or_critical_pressure_rate_pct
0,2023-01-12 00:00:00,1,2023-01-12,2023-01-12,"6,619.00",6619,6619,-402.00,-402,0.00,0.00,0,0.00,0,0.00,0,0.00,0,0.99,0.99,1,0,0,0,0,Normal Operating Pressure,Routine Monitoring,No Major Pressure Driver,Daily,0,0.00,0.00,0,0.00
1,2023-01-22 00:00:00,1,2023-01-22,2023-01-22,"7,171.00",7171,7171,-188.00,-188,8.34,0.00,0,5.00,5,6.00,6,7.00,7,6.07,6.07,0,0,0,0,1,Critical System Pressure,Immediate Escalation Required,Sudden Spike Pressure,Daily,1,100.00,100.00,1,100.00
2,2023-01-23 00:00:00,1,2023-01-23,2023-01-23,"7,330.00",7330,7330,-142.00,-142,2.22,0.00,0,5.00,5,2.00,2,7.00,7,4.70,4.70,0,0,0,0,1,Critical System Pressure,Immediate Escalation Required,Prolonged Window Pressure,Daily,1,100.00,100.00,1,100.00
3,2023-01-24 00:00:00,1,2023-01-24,2023-01-24,"7,475.00",7475,7475,-128.00,-128,1.98,0.00,0,5.00,5,0.00,0,4.00,4,3.49,3.49,0,0,0,1,0,High System Stress,Priority Monitoring Required,Prolonged Window Pressure,Daily,1,100.00,0.00,1,100.00
4,2023-01-25 00:00:00,1,2023-01-25,2023-01-25,"7,560.00",7560,7560,-139.00,-139,1.14,0.00,0,5.00,5,0.00,0,4.00,4,3.48,3.48,0,0,0,1,0,High System Stress,Priority Monitoring Required,Prolonged Window Pressure,Daily,1,100.00,0.00,1,100.00
5,2023-01-29 00:00:00,1,2023-01-29,2023-01-29,"7,517.00",7517,7517,-292.00,-292,-0.57,0.00,0,5.00,5,2.00,2,7.00,7,4.84,4.84,0,0,0,0,1,Critical System Pressure,Immediate Escalation Required,Prolonged Window Pressure,Daily,1,100.00,100.00,1,100.00
6,2023-01-30 00:00:00,1,2023-01-30,2023-01-30,"7,797.00",7797,7797,-167.00,-167,3.72,0.00,0,5.00,5,3.00,3,7.00,7,5.11,5.11,0,0,0,0,1,Critical System Pressure,Immediate Escalation Required,Prolonged Window Pressure,Daily,1,100.00,100.00,1,100.00
7,2023-01-31 00:00:00,1,2023-01-31,2023-01-31,"7,839.00",7839,7839,-122.00,-122,0.54,0.00,0,5.00,5,0.00,0,3.00,3,3.24,3.24,0,0,1,0,0,Moderate System Stress,Operational Watch Required,Prolonged Window Pressure,Daily,1,100.00,0.00,0,0.00
8,2023-02-01 00:00:00,1,2023-02-01,2023-02-01,"7,935.00",7935,7935,-204.00,-204,1.22,0.00,0,4.00,4,0.00,0,3.00,3,3.08,3.08,0,0,1,0,0,Moderate System Stress,Operational Watch Required,Prolonged Window Pressure,Daily,1,100.00,0.00,0,0.00
9,2023-02-02 00:00:00,1,2023-02-02,2023-02-02,"7,892.00",7892,7892,-275.00,-275,-0.54,0.00,0,4.00,4,0.00,0,4.00,4,3.36,3.36,0,0,1,0,0,Moderate System Stress,Operational Watch Required,Prolonged Window Pressure,Daily,1,100.00,0.00,0,0.00



Step 6B completed successfully.
Time-based pressure trend aggregations are ready for Step 6C.


## Step 6C — Analyze Pressure Trend Direction and Movement Patterns

In this step, we analyze the direction and movement behavior of system pressure across daily, weekly, monthly, quarterly, and yearly trend periods.

This step focuses on:

- Comparing each period with its previous period
- Measuring system load movement
- Measuring pressure/stress score movement
- Identifying increasing, decreasing, and stable pressure trends
- Creating executive-level movement pattern labels
- Preparing trend direction datasets for later insight and visualization steps

The goal is to convert time-based pressure aggregations into interpretable pressure movement signals.

In [3]:
# ============================================================
# Step 6C - Analyze Pressure Trend Direction and Movement Patterns
# ============================================================

import pandas as pd
import numpy as np

print("\nStep 6C - Analyze Pressure Trend Direction and Movement Patterns")
print("=" * 78)

# ------------------------------------------------------------
# 1. Validate Step 6B output
# ------------------------------------------------------------

if "df_combined_pressure_trend" not in globals():
    raise NameError(
        "df_combined_pressure_trend is not available. Please run Step 6B before Step 6C."
    )

df_trend_direction = df_combined_pressure_trend.copy()

print("Trend direction working copy created successfully.")
print(f"Input dataset shape: {df_trend_direction.shape[0]} rows x {df_trend_direction.shape[1]} columns")

# ------------------------------------------------------------
# 2. Required column checks
# ------------------------------------------------------------

required_step_6c_columns = [
    "trend_period_type",
    "trend_period",
    "period_start_date",
    "period_end_date",
    "reporting_days",
    "avg_system_load",
]

missing_step_6c_columns = [
    col for col in required_step_6c_columns if col not in df_trend_direction.columns
]

print(f"Missing Step 6C required columns: {len(missing_step_6c_columns)}")

if missing_step_6c_columns:
    raise ValueError(f"Missing required Step 6C columns: {missing_step_6c_columns}")

# ------------------------------------------------------------
# 3. Date parsing and chronological sorting
# ------------------------------------------------------------

df_trend_direction["period_start_date"] = pd.to_datetime(
    df_trend_direction["period_start_date"],
    errors="coerce"
)

df_trend_direction["period_end_date"] = pd.to_datetime(
    df_trend_direction["period_end_date"],
    errors="coerce"
)

date_parse_errors = int(
    df_trend_direction[["period_start_date", "period_end_date"]].isna().sum().sum()
)

df_trend_direction = df_trend_direction.sort_values(
    by=["trend_period_type", "period_start_date", "period_end_date"]
).reset_index(drop=True)

chronological_order_check = (
    df_trend_direction.groupby("trend_period_type")["period_start_date"]
    .apply(lambda x: x.is_monotonic_increasing)
    .all()
)

print(f"Date parse errors: {date_parse_errors}")
print(f"Dataset chronological order by trend period type: {chronological_order_check}")

# ------------------------------------------------------------
# 4. Detect available stress score columns safely
# ------------------------------------------------------------

possible_stress_score_columns = [
    "avg_final_pressure_stress_score",
    "final_pressure_stress_score_mean",
    "avg_pressure_stress_score",
    "pressure_stress_score",
    "avg_daily_strain_score",
    "daily_strain_score",
]

stress_score_column = None

for col in possible_stress_score_columns:
    if col in df_trend_direction.columns:
        stress_score_column = col
        break

if stress_score_column is None:
    print("No average stress score column found. Step 6C will use avg_system_load only for movement logic.")
else:
    print(f"Stress score column detected: {stress_score_column}")

# ------------------------------------------------------------
# 5. Create previous period comparison features
# ------------------------------------------------------------

group_col = "trend_period_type"

df_trend_direction["previous_period"] = (
    df_trend_direction.groupby(group_col)["trend_period"].shift(1)
)

df_trend_direction["previous_avg_system_load"] = (
    df_trend_direction.groupby(group_col)["avg_system_load"].shift(1)
)

df_trend_direction["system_load_change"] = (
    df_trend_direction["avg_system_load"] - df_trend_direction["previous_avg_system_load"]
)

df_trend_direction["system_load_change_pct"] = np.where(
    df_trend_direction["previous_avg_system_load"].replace(0, np.nan).notna(),
    (df_trend_direction["system_load_change"] / df_trend_direction["previous_avg_system_load"]) * 100,
    np.nan
)

if stress_score_column is not None:
    df_trend_direction["previous_stress_score"] = (
        df_trend_direction.groupby(group_col)[stress_score_column].shift(1)
    )

    df_trend_direction["stress_score_change"] = (
        df_trend_direction[stress_score_column] - df_trend_direction["previous_stress_score"]
    )

    df_trend_direction["stress_score_change_pct"] = np.where(
        df_trend_direction["previous_stress_score"].replace(0, np.nan).notna(),
        (df_trend_direction["stress_score_change"] / df_trend_direction["previous_stress_score"]) * 100,
        np.nan
    )
else:
    df_trend_direction["previous_stress_score"] = np.nan
    df_trend_direction["stress_score_change"] = np.nan
    df_trend_direction["stress_score_change_pct"] = np.nan

print("Previous period comparison features created successfully.")

# ------------------------------------------------------------
# 6. Create trend direction labels
# ------------------------------------------------------------

def classify_direction(change_value, stable_threshold):
    if pd.isna(change_value):
        return "No Previous Period"
    elif change_value > stable_threshold:
        return "Increasing"
    elif change_value < -stable_threshold:
        return "Decreasing"
    else:
        return "Stable"


# Dynamic stability threshold based on load volatility
load_change_std = df_trend_direction["system_load_change"].std(skipna=True)

if pd.isna(load_change_std) or load_change_std == 0:
    load_stable_threshold = 0
else:
    load_stable_threshold = round(load_change_std * 0.15, 2)

if stress_score_column is not None:
    stress_change_std = df_trend_direction["stress_score_change"].std(skipna=True)

    if pd.isna(stress_change_std) or stress_change_std == 0:
        stress_stable_threshold = 0
    else:
        stress_stable_threshold = round(stress_change_std * 0.15, 2)
else:
    stress_stable_threshold = np.nan

df_trend_direction["system_load_trend_direction"] = df_trend_direction["system_load_change"].apply(
    lambda x: classify_direction(x, load_stable_threshold)
)

if stress_score_column is not None:
    df_trend_direction["stress_score_trend_direction"] = df_trend_direction["stress_score_change"].apply(
        lambda x: classify_direction(x, stress_stable_threshold)
    )
else:
    df_trend_direction["stress_score_trend_direction"] = "Not Available"

print(f"System load stable threshold used: {load_stable_threshold}")
print(f"Stress score stable threshold used: {stress_stable_threshold}")

# ------------------------------------------------------------
# 7. Create combined pressure movement score
# ------------------------------------------------------------

# Normalize changes safely for comparable movement scoring
load_change_abs_max = df_trend_direction["system_load_change"].abs().max(skipna=True)

if pd.isna(load_change_abs_max) or load_change_abs_max == 0:
    df_trend_direction["normalized_load_movement"] = 0
else:
    df_trend_direction["normalized_load_movement"] = (
        df_trend_direction["system_load_change"] / load_change_abs_max
    )

if stress_score_column is not None:
    stress_change_abs_max = df_trend_direction["stress_score_change"].abs().max(skipna=True)

    if pd.isna(stress_change_abs_max) or stress_change_abs_max == 0:
        df_trend_direction["normalized_stress_movement"] = 0
    else:
        df_trend_direction["normalized_stress_movement"] = (
            df_trend_direction["stress_score_change"] / stress_change_abs_max
        )

    df_trend_direction["pressure_movement_score"] = (
        (df_trend_direction["normalized_load_movement"].fillna(0) * 0.55)
        + (df_trend_direction["normalized_stress_movement"].fillna(0) * 0.45)
    )
else:
    df_trend_direction["normalized_stress_movement"] = np.nan
    df_trend_direction["pressure_movement_score"] = df_trend_direction["normalized_load_movement"].fillna(0)

# ------------------------------------------------------------
# 8. Classify final pressure trend movement
# ------------------------------------------------------------

movement_score_std = df_trend_direction["pressure_movement_score"].std(skipna=True)

if pd.isna(movement_score_std) or movement_score_std == 0:
    movement_stable_threshold = 0
else:
    movement_stable_threshold = round(movement_score_std * 0.20, 4)

def classify_pressure_movement(row):
    if pd.isna(row["previous_period"]):
        return "No Previous Period"
    elif row["pressure_movement_score"] > movement_stable_threshold:
        return "Pressure Increasing"
    elif row["pressure_movement_score"] < -movement_stable_threshold:
        return "Pressure Decreasing"
    else:
        return "Pressure Stable"


df_trend_direction["pressure_trend_direction"] = df_trend_direction.apply(
    classify_pressure_movement,
    axis=1
)

print(f"Pressure movement stable threshold used: {movement_stable_threshold}")

# ------------------------------------------------------------
# 9. Create executive movement pattern labels
# ------------------------------------------------------------

def classify_movement_pattern(row):
    load_direction = row["system_load_trend_direction"]
    stress_direction = row["stress_score_trend_direction"]
    final_direction = row["pressure_trend_direction"]

    if final_direction == "No Previous Period":
        return "Baseline Period"

    if load_direction == "Increasing" and stress_direction == "Increasing":
        return "Broad Pressure Build-up"

    if load_direction == "Increasing" and stress_direction in ["Stable", "Not Available"]:
        return "Load-Led Pressure Increase"

    if load_direction in ["Stable", "Decreasing"] and stress_direction == "Increasing":
        return "Stress-Led Pressure Increase"

    if load_direction == "Decreasing" and stress_direction == "Decreasing":
        return "Broad Pressure Relief"

    if final_direction == "Pressure Increasing":
        return "Pressure Build-up"

    if final_direction == "Pressure Decreasing":
        return "Pressure Cooling"

    return "Stable Pressure Movement"


df_trend_direction["pressure_movement_pattern"] = df_trend_direction.apply(
    classify_movement_pattern,
    axis=1
)

# ------------------------------------------------------------
# 10. Create executive pressure signal
# ------------------------------------------------------------

def classify_executive_signal(row):
    direction = row["pressure_trend_direction"]
    pattern = row["pressure_movement_pattern"]

    if direction == "No Previous Period":
        return "Baseline"

    if pattern in ["Broad Pressure Build-up", "Stress-Led Pressure Increase"]:
        return "Watch Closely"

    if pattern in ["Load-Led Pressure Increase", "Pressure Build-up"]:
        return "Monitor Increase"

    if pattern in ["Broad Pressure Relief", "Pressure Cooling"]:
        return "Improving"

    return "Stable"


df_trend_direction["executive_pressure_signal"] = df_trend_direction.apply(
    classify_executive_signal,
    axis=1
)

print("Pressure trend direction and movement pattern labels created successfully.")

# ------------------------------------------------------------
# 11. Create summary tables
# ------------------------------------------------------------

df_pressure_direction_summary = (
    df_trend_direction.groupby(["trend_period_type", "pressure_trend_direction"])
    .size()
    .reset_index(name="record_count")
)

df_pressure_direction_summary["record_percentage"] = (
    df_pressure_direction_summary["record_count"]
    / df_pressure_direction_summary.groupby("trend_period_type")["record_count"].transform("sum")
    * 100
).round(2)

df_movement_pattern_summary = (
    df_trend_direction.groupby(["trend_period_type", "pressure_movement_pattern"])
    .size()
    .reset_index(name="record_count")
)

df_movement_pattern_summary["record_percentage"] = (
    df_movement_pattern_summary["record_count"]
    / df_movement_pattern_summary.groupby("trend_period_type")["record_count"].transform("sum")
    * 100
).round(2)

df_executive_signal_summary = (
    df_trend_direction.groupby(["trend_period_type", "executive_pressure_signal"])
    .size()
    .reset_index(name="record_count")
)

df_executive_signal_summary["record_percentage"] = (
    df_executive_signal_summary["record_count"]
    / df_executive_signal_summary.groupby("trend_period_type")["record_count"].transform("sum")
    * 100
).round(2)

print("Step 6C summary tables created successfully.")

# ------------------------------------------------------------
# 12. Step 6C validation summary
# ------------------------------------------------------------

new_step_6c_columns = [
    "previous_period",
    "previous_avg_system_load",
    "system_load_change",
    "system_load_change_pct",
    "previous_stress_score",
    "stress_score_change",
    "stress_score_change_pct",
    "system_load_trend_direction",
    "stress_score_trend_direction",
    "normalized_load_movement",
    "normalized_stress_movement",
    "pressure_movement_score",
    "pressure_trend_direction",
    "pressure_movement_pattern",
    "executive_pressure_signal",
]

missing_new_step_6c_columns = [
    col for col in new_step_6c_columns if col not in df_trend_direction.columns
]

missing_values_core_trend_cols = int(
    df_trend_direction[
        [
            "pressure_movement_score",
            "pressure_trend_direction",
            "pressure_movement_pattern",
            "executive_pressure_signal",
        ]
    ].isna().sum().sum()
)

df_step_6c_validation_summary = pd.DataFrame({
    "Step 6C Check": [
        "Input Dataset Source",
        "Rows Checked",
        "Columns Before Step 6C",
        "Columns After Step 6C",
        "New Trend Direction Columns Created",
        "Missing Required Input Columns",
        "Missing New Step 6C Columns",
        "Date Parse Errors",
        "Chronological Order",
        "Missing Values in Core Trend Output Columns",
        "Trend Period Types Available",
        "Stress Score Column Used",
        "Rows Deleted or Modified",
    ],
    "Result": [
        "df_combined_pressure_trend",
        len(df_trend_direction),
        df_combined_pressure_trend.shape[1],
        df_trend_direction.shape[1],
        len(new_step_6c_columns),
        len(missing_step_6c_columns),
        len(missing_new_step_6c_columns),
        date_parse_errors,
        chronological_order_check,
        missing_values_core_trend_cols,
        df_trend_direction["trend_period_type"].nunique(),
        stress_score_column if stress_score_column is not None else "Not Available",
        0,
    ]
})

# ------------------------------------------------------------
# 13. Display outputs
# ------------------------------------------------------------

print("\nStep 6C validation summary:")
display(df_step_6c_validation_summary)

print("\nPressure trend direction summary:")
display(df_pressure_direction_summary)

print("\nPressure movement pattern summary:")
display(df_movement_pattern_summary)

print("\nExecutive pressure signal summary:")
display(df_executive_signal_summary)

preview_columns = [
    "trend_period_type",
    "trend_period",
    "period_start_date",
    "period_end_date",
    "reporting_days",
    "avg_system_load",
    "previous_avg_system_load",
    "system_load_change",
    "system_load_change_pct",
    "pressure_movement_score",
    "pressure_trend_direction",
    "pressure_movement_pattern",
    "executive_pressure_signal",
]

available_preview_columns = [
    col for col in preview_columns if col in df_trend_direction.columns
]

print("\nTrend direction dataset preview:")
display(df_trend_direction[available_preview_columns].head(10))

print("\nTrend direction dataset tail:")
display(df_trend_direction[available_preview_columns].tail(10))

# ------------------------------------------------------------
# 14. Completion message
# ------------------------------------------------------------

print("\nStep 6C completed successfully.")
print("Pressure trend direction and movement pattern analysis is ready for Step 6D.")


Step 6C - Analyze Pressure Trend Direction and Movement Patterns
Trend direction working copy created successfully.
Input dataset shape: 925 rows x 34 columns
Missing Step 6C required columns: 0
Date parse errors: 0
Dataset chronological order by trend period type: True
Stress score column detected: avg_final_pressure_stress_score
Previous period comparison features created successfully.
System load stable threshold used: 51.62
Stress score stable threshold used: 0.18
Pressure movement stable threshold used: 0.0225
Pressure trend direction and movement pattern labels created successfully.
Step 6C summary tables created successfully.

Step 6C validation summary:


,Step 6C Check,Result
0,Input Dataset Source,df_combined_pressure_trend
1,Rows Checked,925
2,Columns Before Step 6C,34
3,Columns After Step 6C,49
4,New Trend Direction Columns Created,15
5,Missing Required Input Columns,0
6,Missing New Step 6C Columns,0
7,Date Parse Errors,0
8,Chronological Order,True
9,Missing Values in Core Trend Output Columns,0



Pressure trend direction summary:


,trend_period_type,pressure_trend_direction,record_count,record_percentage
0,Daily,No Previous Period,1,0.14
1,Daily,Pressure Decreasing,283,39.31
2,Daily,Pressure Increasing,230,31.94
3,Daily,Pressure Stable,206,28.61
4,Monthly,No Previous Period,1,2.78
5,Monthly,Pressure Decreasing,16,44.44
6,Monthly,Pressure Increasing,12,33.33
7,Monthly,Pressure Stable,7,19.44
8,Quarterly,No Previous Period,1,8.33
9,Quarterly,Pressure Decreasing,7,58.33



Pressure movement pattern summary:


...


Trend direction dataset preview:


,trend_period_type,trend_period,period_start_date,period_end_date,reporting_days,avg_system_load,previous_avg_system_load,system_load_change,system_load_change_pct,pressure_movement_score,pressure_trend_direction,pressure_movement_pattern,executive_pressure_signal
0,Daily,2023-01-12 00:00:00,2023-01-12,2023-01-12,1,"6,619.00",NaN,NaN,NaN,0.00,No Previous Period,Baseline Period,Baseline
1,Daily,2023-01-22 00:00:00,2023-01-22,2023-01-22,1,"7,171.00","6,619.00",552.00,8.34,0.51,Pressure Increasing,Broad Pressure Build-up,Watch Closely
2,Daily,2023-01-23 00:00:00,2023-01-23,2023-01-23,1,"7,330.00","7,171.00",159.00,2.22,-0.10,Pressure Decreasing,Pressure Cooling,Improving
3,Daily,2023-01-24 00:00:00,2023-01-24,2023-01-24,1,"7,475.00","7,330.00",145.00,1.98,-0.09,Pressure Decreasing,Pressure Cooling,Improving
4,Daily,2023-01-25 00:00:00,2023-01-25,2023-01-25,1,"7,560.00","7,475.00",85.00,1.14,0.01,Pressure Stable,Load-Led Pressure Increase,Monitor Increase
5,Daily,2023-01-29 00:00:00,2023-01-29,2023-01-29,1,"7,517.00","7,560.00",-43.00,-0.57,0.12,Pressure Increasing,Stress-Led Pressure Increase,Watch Closely
6,Daily,2023-01-30 00:00:00,2023-01-30,2023-01-30,1,"7,797.00","7,517.00",280.00,3.72,0.06,Pressure Increasing,Broad Pressure Build-up,Watch Closely
7,Daily,2023-01-31 00:00:00,2023-01-31,2023-01-31,1,"7,839.00","7,797.00",42.00,0.54,-0.16,Pressure Decreasing,Pressure Cooling,Improving
8,Daily,2023-02-01 00:00:00,2023-02-01,2023-02-01,1,"7,935.00","7,839.00",96.00,1.22,-0.00,Pressure Stable,Load-Led Pressure Increase,Monitor Increase
9,Daily,2023-02-02 00:00:00,2023-02-02,2023-02-02,1,"7,892.00","7,935.00",-43.00,-0.54,0.02,Pressure Stable,Stress-Led Pressure Increase,Watch Closely



Trend direction dataset tail:


,trend_period_type,trend_period,period_start_date,period_end_date,reporting_days,avg_system_load,previous_avg_system_load,system_load_change,system_load_change_pct,pressure_movement_score,pressure_trend_direction,pressure_movement_pattern,executive_pressure_signal
915,Weekly,2025-11-03/2025-11-09,2025-11-03,2025-11-09,5,"2,307.40","2,305.60",1.80,0.08,-0.01,Pressure Stable,Stable Pressure Movement,Stable
916,Weekly,2025-11-10/2025-11-16,2025-11-11,2025-11-16,4,"2,347.00","2,307.40",39.60,1.72,0.02,Pressure Stable,Stable Pressure Movement,Stable
917,Weekly,2025-11-17/2025-11-23,2025-11-17,2025-11-23,5,"2,390.20","2,347.00",43.20,1.84,-0.01,Pressure Stable,Stable Pressure Movement,Stable
918,Weekly,2025-11-24/2025-11-30,2025-11-24,2025-11-30,4,"2,419.75","2,390.20",29.55,1.24,0.09,Pressure Increasing,Stress-Led Pressure Increase,Watch Closely
919,Weekly,2025-12-01/2025-12-07,2025-12-01,2025-12-07,5,"2,447.60","2,419.75",27.85,1.15,-0.06,Pressure Decreasing,Pressure Cooling,Improving
920,Weekly,2025-12-08/2025-12-14,2025-12-08,2025-12-14,5,"2,482.80","2,447.60",35.20,1.44,-0.04,Pressure Decreasing,Pressure Cooling,Improving
921,Weekly,2025-12-15/2025-12-21,2025-12-15,2025-12-21,5,"2,514.00","2,482.80",31.20,1.26,0.08,Pressure Increasing,Stress-Led Pressure Increase,Watch Closely
922,Yearly,2023,2023-01-12,2023-12-28,230,"8,846.54",NaN,NaN,NaN,0.00,No Previous Period,Baseline Period,Baseline
923,Yearly,2024,2024-01-01,2024-12-30,251,"7,319.86","8,846.54","-1,526.68",-17.26,-0.20,Pressure Decreasing,Broad Pressure Relief,Improving
924,Yearly,2025,2025-01-01,2025-12-21,239,"2,575.74","7,319.86","-4,744.12",-64.81,-0.67,Pressure Decreasing,Broad Pressure Relief,Improving



Step 6C completed successfully.
Pressure trend direction and movement pattern analysis is ready for Step 6D.


## Step 6D — Build Final KPI Summary Layer

In this step, we build the final KPI summary layer required by the project guidelines.

This step converts the final daily pressure and stress classification dataset into a clean executive KPI table. These KPIs summarize system-wide care load, intake pressure, volatility, backlog pressure, and discharge offset capacity.

### Required KPIs

- Total Children Under Care
- Net Intake Pressure
- Care Load Volatility Index
- Backlog Accumulation Rate
- Discharge Offset Ratio

### Purpose

The purpose of this step is to create a dashboard-ready KPI summary layer that can be used for:

- KPI summary cards
- Executive reporting
- Streamlit dashboard modules
- Notebook 07 final business insights and recommendations

This step does not repeat pressure window detection or trend movement analysis. It only creates the final KPI reporting layer.

In [4]:
# ============================================================
# Step 6D - Build Final KPI Summary Layer
# ============================================================

import pandas as pd
import numpy as np

print("Step 6D - Build Final KPI Summary Layer")
print("=" * 70)

# ------------------------------------------------------------
# 1. Validate Notebook 06 base dataset
# ------------------------------------------------------------

if "df_notebook_06" not in globals():
    raise NameError(
        "df_notebook_06 not found. Please run Step 6A before Step 6D."
    )

df_kpi_base = df_notebook_06.copy()

print("KPI base working copy created successfully.")
print(f"Input dataset shape: {df_kpi_base.shape[0]} rows x {df_kpi_base.shape[1]} columns")

# ------------------------------------------------------------
# 2. Required column validation
# ------------------------------------------------------------

required_step_6d_columns = [
    "Date",
    "total_system_load",
    "net_daily_intake",
    "care_load_growth_rate_pct",
    "backlog_severity_score",
    "final_pressure_stress_score",
    "final_pressure_stress_level",
    "operational_response_category",
    "primary_pressure_driver",
    "Children in CBP custody",
    "Children in HHS Care",
    "Children transferred out of CBP custody",
    "Children discharged from HHS Care"
]

missing_step_6d_columns = [
    col for col in required_step_6d_columns
    if col not in df_kpi_base.columns
]

print(f"Missing Step 6D required columns: {len(missing_step_6d_columns)}")

if missing_step_6d_columns:
    raise ValueError(
        f"Missing required Step 6D columns: {missing_step_6d_columns}"
    )

# ------------------------------------------------------------
# 3. Date validation and chronological sorting
# ------------------------------------------------------------

df_kpi_base["Date"] = pd.to_datetime(df_kpi_base["Date"], errors="coerce")

date_parse_errors_6d = int(df_kpi_base["Date"].isna().sum())

if date_parse_errors_6d > 0:
    raise ValueError(
        f"Date parsing issue found in Step 6D. Invalid Date rows: {date_parse_errors_6d}"
    )

df_kpi_base = df_kpi_base.sort_values("Date").reset_index(drop=True)

duplicate_dates_6d = int(df_kpi_base["Date"].duplicated().sum())
chronological_order_6d = bool(df_kpi_base["Date"].is_monotonic_increasing)

print(f"Date parse errors: {date_parse_errors_6d}")
print(f"Duplicate dates: {duplicate_dates_6d}")
print(f"Dataset chronological order: {chronological_order_6d}")

# ------------------------------------------------------------
# 4. Confirm numeric KPI fields
# ------------------------------------------------------------

numeric_kpi_columns = [
    "total_system_load",
    "net_daily_intake",
    "care_load_growth_rate_pct",
    "backlog_severity_score",
    "final_pressure_stress_score",
    "Children in CBP custody",
    "Children in HHS Care",
    "Children transferred out of CBP custody",
    "Children discharged from HHS Care"
]

for col in numeric_kpi_columns:
    df_kpi_base[col] = pd.to_numeric(df_kpi_base[col], errors="coerce")

numeric_kpi_validation = pd.DataFrame({
    "KPI Input Column": numeric_kpi_columns,
    "Missing Values": [int(df_kpi_base[col].isna().sum()) for col in numeric_kpi_columns],
    "Minimum": [df_kpi_base[col].min() for col in numeric_kpi_columns],
    "Maximum": [df_kpi_base[col].max() for col in numeric_kpi_columns],
    "Data Type": [str(df_kpi_base[col].dtype) for col in numeric_kpi_columns]
})

print("\nNumeric KPI input validation:")
display(numeric_kpi_validation)

# ------------------------------------------------------------
# 5. Calculate final KPI values
# ------------------------------------------------------------

total_reporting_days = len(df_kpi_base)

# KPI 1: Total Children Under Care
# Guideline meaning: system-wide responsibility
total_children_under_care_current = int(df_kpi_base["total_system_load"].iloc[-1])
total_children_under_care_average = round(df_kpi_base["total_system_load"].mean(), 2)
total_children_under_care_peak = int(df_kpi_base["total_system_load"].max())
total_children_under_care_minimum = int(df_kpi_base["total_system_load"].min())

# KPI 2: Net Intake Pressure
# Guideline meaning: inflow vs outflow imbalance
net_intake_pressure_total = int(df_kpi_base["net_daily_intake"].sum())
net_intake_pressure_average = round(df_kpi_base["net_daily_intake"].mean(), 2)
positive_net_intake_days = int((df_kpi_base["net_daily_intake"] > 0).sum())
negative_net_intake_days = int((df_kpi_base["net_daily_intake"] < 0).sum())
balanced_net_intake_days = int((df_kpi_base["net_daily_intake"] == 0).sum())
net_intake_pressure_rate_pct = round(
    positive_net_intake_days / total_reporting_days * 100,
    2
)

# KPI 3: Care Load Volatility Index
# Guideline meaning: stability of system
care_load_volatility_index = round(df_kpi_base["care_load_growth_rate_pct"].std(), 2)
system_load_std = round(df_kpi_base["total_system_load"].std(), 2)
average_absolute_growth_change = round(df_kpi_base["care_load_growth_rate_pct"].abs().mean(), 2)

# KPI 4: Backlog Accumulation Rate
# Guideline meaning: sustained care pressure
backlog_pressure_days = int((df_kpi_base["backlog_severity_score"] > 0).sum())
sustained_backlog_days = int((df_kpi_base["backlog_severity_score"] >= 2).sum())
critical_backlog_days = int((df_kpi_base["backlog_severity_score"] >= 3).sum())
backlog_accumulation_rate_pct = round(
    sustained_backlog_days / total_reporting_days * 100,
    2
)

# KPI 5: Discharge Offset Ratio
# Guideline meaning: ability to relieve load
total_transfers_to_hhs = int(df_kpi_base["Children transferred out of CBP custody"].sum())
total_discharges_from_hhs = int(df_kpi_base["Children discharged from HHS Care"].sum())

discharge_offset_ratio = round(
    total_discharges_from_hhs / total_transfers_to_hhs,
    2
) if total_transfers_to_hhs != 0 else np.nan

discharge_offset_pct = round(
    discharge_offset_ratio * 100,
    2
) if pd.notna(discharge_offset_ratio) else np.nan

discharge_surplus_or_gap = int(total_discharges_from_hhs - total_transfers_to_hhs)

# Additional executive pressure context
critical_pressure_days = int(
    (df_kpi_base["final_pressure_stress_level"] == "Critical System Pressure").sum()
)

high_pressure_days = int(
    (df_kpi_base["final_pressure_stress_level"] == "High System Stress").sum()
)

moderate_pressure_days = int(
    (df_kpi_base["final_pressure_stress_level"] == "Moderate System Stress").sum()
)

average_final_pressure_score = round(df_kpi_base["final_pressure_stress_score"].mean(), 2)
maximum_final_pressure_score = round(df_kpi_base["final_pressure_stress_score"].max(), 2)

# ------------------------------------------------------------
# 6. Create final KPI summary layer
# ------------------------------------------------------------

df_final_kpi_summary = pd.DataFrame({
    "KPI Name": [
        "Total Children Under Care",
        "Net Intake Pressure",
        "Care Load Volatility Index",
        "Backlog Accumulation Rate",
        "Discharge Offset Ratio"
    ],
    "KPI Description": [
        "System-wide responsibility measured through total CBP + HHS care load.",
        "Inflow vs outflow imbalance measured through transfers into HHS minus discharges from HHS.",
        "System stability measured through variability in care load growth rate.",
        "Sustained care pressure measured through sustained backlog severity days.",
        "Ability to relieve load measured by discharges from HHS relative to transfers into HHS."
    ],
    "Primary Value": [
        total_children_under_care_current,
        net_intake_pressure_average,
        care_load_volatility_index,
        backlog_accumulation_rate_pct,
        discharge_offset_ratio
    ],
    "Primary Unit": [
        "Children",
        "Avg Net Intake / Day",
        "Std Dev of Growth Rate (%)",
        "% Sustained Backlog Days",
        "Discharges / Transfers"
    ],
    "Supporting Value 1": [
        total_children_under_care_average,
        net_intake_pressure_total,
        system_load_std,
        backlog_pressure_days,
        discharge_offset_pct
    ],
    "Supporting Label 1": [
        "Average Total System Load",
        "Total Net Intake",
        "System Load Standard Deviation",
        "Any Backlog Pressure Days",
        "Discharge Offset Percentage"
    ],
    "Supporting Value 2": [
        total_children_under_care_peak,
        positive_net_intake_days,
        average_absolute_growth_change,
        sustained_backlog_days,
        discharge_surplus_or_gap
    ],
    "Supporting Label 2": [
        "Peak Total System Load",
        "Positive Net Intake Days",
        "Average Absolute Growth Change",
        "Sustained Backlog Days",
        "Discharge Surplus / Gap"
    ],
    "Supporting Value 3": [
        total_children_under_care_minimum,
        net_intake_pressure_rate_pct,
        maximum_final_pressure_score,
        critical_backlog_days,
        total_discharges_from_hhs
    ],
    "Supporting Label 3": [
        "Minimum Total System Load",
        "Positive Net Intake Day Rate (%)",
        "Maximum Final Pressure Score",
        "Critical Backlog Days",
        "Total Discharges from HHS"
    ],
    "Dashboard Module": [
        "KPI Summary Cards / System Load Overview",
        "Net Intake & Backlog Trends",
        "KPI Summary Cards / Pressure Monitoring",
        "Net Intake & Backlog Trends",
        "CBP vs HHS Load Comparison"
    ],
    "Business Interpretation": [
        "Shows the current system-wide care responsibility across CBP and HHS.",
        "Shows whether incoming transfers are exceeding discharge relief capacity.",
        "Shows how stable or unstable care load movement is over time.",
        "Shows how frequently backlog pressure becomes sustained rather than short-term.",
        "Shows whether HHS discharges are offsetting transfers into the care system."
    ]
})

print("\nFinal KPI summary layer created successfully:")
display(df_final_kpi_summary)

# ------------------------------------------------------------
# 7. Create KPI dashboard card summary
# ------------------------------------------------------------

df_kpi_dashboard_cards = pd.DataFrame({
    "Card Title": [
        "Current Children Under Care",
        "Average Net Intake Pressure",
        "Care Load Volatility Index",
        "Backlog Accumulation Rate",
        "Discharge Offset Ratio",
        "Critical Pressure Days",
        "High Stress Days",
        "Average Final Pressure Score"
    ],
    "Card Value": [
        total_children_under_care_current,
        net_intake_pressure_average,
        care_load_volatility_index,
        backlog_accumulation_rate_pct,
        discharge_offset_ratio,
        critical_pressure_days,
        high_pressure_days,
        average_final_pressure_score
    ],
    "Card Unit": [
        "Children",
        "Children / Day",
        "Growth Std Dev",
        "%",
        "Ratio",
        "Days",
        "Days",
        "Score"
    ],
    "Card Context": [
        "Latest total system load.",
        "Average inflow-outflow imbalance.",
        "Volatility in care load growth.",
        "Share of days with sustained backlog pressure.",
        "Discharges compared with transfers into HHS.",
        "Days requiring immediate escalation.",
        "Days requiring priority monitoring.",
        "Overall pressure score average."
    ]
})

print("\nKPI dashboard card summary:")
display(df_kpi_dashboard_cards)

# ------------------------------------------------------------
# 8. Create KPI validation summary
# ------------------------------------------------------------

step_6d_validation_summary = pd.DataFrame({
    "Step 6D Check": [
        "Input Dataset Source",
        "Rows Checked",
        "Columns Checked",
        "Missing Required Input Columns",
        "Date Parse Errors",
        "Duplicate Dates",
        "Chronological Order",
        "Final KPI Count",
        "Dashboard KPI Card Count",
        "Rows Deleted or Modified"
    ],
    "Result": [
        "df_notebook_06",
        df_kpi_base.shape[0],
        df_kpi_base.shape[1],
        len(missing_step_6d_columns),
        date_parse_errors_6d,
        duplicate_dates_6d,
        chronological_order_6d,
        df_final_kpi_summary.shape[0],
        df_kpi_dashboard_cards.shape[0],
        0
    ]
})

print("\nStep 6D validation summary:")
display(step_6d_validation_summary)

# ------------------------------------------------------------
# 9. Save working outputs in memory for next steps
# ------------------------------------------------------------

df_step_6d_kpi_base = df_kpi_base.copy()

print("\nStep 6D completed successfully.")
print("Final KPI summary layer is ready for Step 6E.")

Step 6D - Build Final KPI Summary Layer
KPI base working copy created successfully.
Input dataset shape: 720 rows x 107 columns
Missing Step 6D required columns: 0
Date parse errors: 0
Duplicate dates: 0
Dataset chronological order: True

Numeric KPI input validation:


,KPI Input Column,Missing Values,Minimum,Maximum,Data Type
0,total_system_load,0,"2,002.00","11,762.00",int64
1,net_daily_intake,0,-465.00,206.00,int64
2,care_load_growth_rate_pct,0,-11.60,8.34,float64
3,backlog_severity_score,0,0.00,3.00,int64
4,final_pressure_stress_score,0,0.01,6.07,float64
5,Children in CBP custody,0,7.00,531.00,int64
6,Children in HHS Care,0,"1,972.00","11,516.00",int64
7,Children transferred out of CBP custody,0,0.00,440.00,int64
8,Children discharged from HHS Care,0,0.00,505.00,int64



Final KPI summary layer created successfully:


,KPI Name,KPI Description,Primary Value,Primary Unit,Supporting Value 1,Supporting Label 1,Supporting Value 2,Supporting Label 2,Supporting Value 3,Supporting Label 3,Dashboard Module,Business Interpretation
0,Total Children Under Care,System-wide responsibility measured through to...,"2,502.00",Children,"6,232.77",Average Total System Load,"11,762.00",Peak Total System Load,"2,002.00",Minimum Total System Load,KPI Summary Cards / System Load Overview,Shows the current system-wide care responsibil...
1,Net Intake Pressure,Inflow vs outflow imbalance measured through t...,-44.74,Avg Net Intake / Day,"-32,212.00",Total Net Intake,238.00,Positive Net Intake Days,33.06,Positive Net Intake Day Rate (%),Net Intake & Backlog Trends,Shows whether incoming transfers are exceeding...
2,Care Load Volatility Index,System stability measured through variability ...,1.89,Std Dev of Growth Rate (%),"2,918.46",System Load Standard Deviation,1.24,Average Absolute Growth Change,6.07,Maximum Final Pressure Score,KPI Summary Cards / Pressure Monitoring,Shows how stable or unstable care load movemen...
3,Backlog Accumulation Rate,Sustained care pressure measured through susta...,13.33,% Sustained Backlog Days,238.00,Any Backlog Pressure Days,96.00,Sustained Backlog Days,27.00,Critical Backlog Days,Net Intake & Backlog Trends,Shows how frequently backlog pressure becomes ...
4,Discharge Offset Ratio,Ability to relieve load measured by discharges...,1.35,Discharges / Transfers,135.00,Discharge Offset Percentage,"32,212.00",Discharge Surplus / Gap,"124,853.00",Total Discharges from HHS,CBP vs HHS Load Comparison,Shows whether HHS discharges are offsetting tr...



KPI dashboard card summary:


,Card Title,Card Value,Card Unit,Card Context
0,Current Children Under Care,"2,502.00",Children,Latest total system load.
1,Average Net Intake Pressure,-44.74,Children / Day,Average inflow-outflow imbalance.
2,Care Load Volatility Index,1.89,Growth Std Dev,Volatility in care load growth.
3,Backlog Accumulation Rate,13.33,%,Share of days with sustained backlog pressure.
4,Discharge Offset Ratio,1.35,Ratio,Discharges compared with transfers into HHS.
5,Critical Pressure Days,78.00,Days,Days requiring immediate escalation.
6,High Stress Days,138.00,Days,Days requiring priority monitoring.
7,Average Final Pressure Score,2.05,Score,Overall pressure score average.



Step 6D validation summary:


,Step 6D Check,Result
0,Input Dataset Source,df_notebook_06
1,Rows Checked,720
2,Columns Checked,107
3,Missing Required Input Columns,0
4,Date Parse Errors,0
5,Duplicate Dates,0
6,Chronological Order,True
7,Final KPI Count,5
8,Dashboard KPI Card Count,8
9,Rows Deleted or Modified,0



Step 6D completed successfully.
Final KPI summary layer is ready for Step 6E.


## Step 6E — Export Final KPI Summary and Dashboard-Ready Analytical Outputs

This step exports the final KPI summary layer and dashboard-ready analytical outputs created in Notebook 06.

The purpose of this step is to make the final trend analysis outputs reusable for:
- Final business insight documentation in Notebook 07
- Future dashboard development
- Executive KPI cards
- Pressure movement and trend interpretation
- Final project-level reporting

This step exports:
- Final KPI summary layer
- KPI dashboard card summary
- Trend direction dataset
- Pressure trend direction summary
- Pressure movement pattern summary
- Executive pressure signal summary
- Step 6D validation summary

This step does not modify the original dataset. It only validates available outputs and saves clean CSV files into the processed data folder.

In [9]:
# ============================================================
# Step 6E - Export Final KPI Summary and Dashboard-Ready Analytical Outputs
# ============================================================

import os
import pandas as pd
import numpy as np

print("Step 6E - Export Final KPI Summary and Dashboard-Ready Analytical Outputs")
print("=" * 75)

# ------------------------------------------------------------
# 1. Define processed output directory
# ------------------------------------------------------------

processed_output_dir = os.path.join("data", "processed")
os.makedirs(processed_output_dir, exist_ok=True)

print("Processed output directory ready:", processed_output_dir)

# ------------------------------------------------------------
# Step 6E Safe Patch - Recreate Missing Step 6C Summary Objects
# ------------------------------------------------------------

print("Step 6E object name alignment and recovery patch started.")

# 1. Try common possible object-name aliases first
if "pressure_trend_direction_summary" not in globals():
    for possible_name in [
        "df_pressure_trend_direction_summary",
        "trend_direction_summary",
        "df_trend_direction_summary",
        "step_6c_trend_direction_summary"
    ]:
        if possible_name in globals():
            pressure_trend_direction_summary = globals()[possible_name].copy()
            break

if "pressure_movement_pattern_summary" not in globals():
    for possible_name in [
        "df_pressure_movement_pattern_summary",
        "movement_pattern_summary",
        "df_movement_pattern_summary",
        "step_6c_movement_pattern_summary"
    ]:
        if possible_name in globals():
            pressure_movement_pattern_summary = globals()[possible_name].copy()
            break

if "executive_pressure_signal_summary" not in globals():
    for possible_name in [
        "df_executive_pressure_signal_summary",
        "executive_signal_summary",
        "df_executive_signal_summary",
        "step_6c_executive_signal_summary"
    ]:
        if possible_name in globals():
            executive_pressure_signal_summary = globals()[possible_name].copy()
            break


# 2. If summary objects are still missing, recreate them from df_trend_direction
if "df_trend_direction" in globals():

    if "pressure_trend_direction_summary" not in globals():
        if {"trend_period_type", "pressure_trend_direction"}.issubset(df_trend_direction.columns):
            pressure_trend_direction_summary = (
                df_trend_direction
                .groupby(["trend_period_type", "pressure_trend_direction"], dropna=False)
                .size()
                .reset_index(name="record_count")
            )

            pressure_trend_direction_summary["record_percentage"] = (
                pressure_trend_direction_summary
                .groupby("trend_period_type")["record_count"]
                .transform(lambda x: round((x / x.sum()) * 100, 2))
            )

    if "pressure_movement_pattern_summary" not in globals():
        if {"trend_period_type", "pressure_movement_pattern"}.issubset(df_trend_direction.columns):
            pressure_movement_pattern_summary = (
                df_trend_direction
                .groupby(["trend_period_type", "pressure_movement_pattern"], dropna=False)
                .size()
                .reset_index(name="record_count")
            )

            pressure_movement_pattern_summary["record_percentage"] = (
                pressure_movement_pattern_summary
                .groupby("trend_period_type")["record_count"]
                .transform(lambda x: round((x / x.sum()) * 100, 2))
            )

    if "executive_pressure_signal_summary" not in globals():
        if {"trend_period_type", "executive_pressure_signal"}.issubset(df_trend_direction.columns):
            executive_pressure_signal_summary = (
                df_trend_direction
                .groupby(["trend_period_type", "executive_pressure_signal"], dropna=False)
                .size()
                .reset_index(name="record_count")
            )

            executive_pressure_signal_summary["record_percentage"] = (
                executive_pressure_signal_summary
                .groupby("trend_period_type")["record_count"]
                .transform(lambda x: round((x / x.sum()) * 100, 2))
            )

# 3. Final object availability check after recovery
recovered_step_6e_objects = {
    "pressure_trend_direction_summary": "pressure_trend_direction_summary" in globals(),
    "pressure_movement_pattern_summary": "pressure_movement_pattern_summary" in globals(),
    "executive_pressure_signal_summary": "executive_pressure_signal_summary" in globals()
}

print("Step 6E recovered object status:")
display(pd.DataFrame(
    recovered_step_6e_objects.items(),
    columns=["Required Object", "Available"]
))

print("Step 6E object name alignment and recovery patch completed.")

# ------------------------------------------------------------
# 2. Validate required Step 6D / Step 6C output objects
# ------------------------------------------------------------

required_step_6e_objects = {
    "df_step_6d_kpi_base": "Final KPI summary layer from Step 6D",
    "df_kpi_dashboard_cards": "Dashboard-ready KPI card summary from Step 6D",
    "df_trend_direction": "Trend direction dataset from Step 6C",
    "pressure_trend_direction_summary": "Pressure trend direction summary from Step 6C",
    "pressure_movement_pattern_summary": "Pressure movement pattern summary from Step 6C",
    "executive_pressure_signal_summary": "Executive pressure signal summary from Step 6C",
    "step_6d_validation_summary": "Step 6D validation summary"
}

missing_step_6e_objects = []

for object_name, object_description in required_step_6e_objects.items():
    if object_name not in globals():
        missing_step_6e_objects.append(object_name)

print("Missing Step 6E required objects:", len(missing_step_6e_objects))

if missing_step_6e_objects:
    raise ValueError(
        "Step 6E cannot continue because these required objects are missing: "
        + ", ".join(missing_step_6e_objects)
    )

print("All required Step 6E objects are available.")


# ------------------------------------------------------------
# 3. Create safe export copies
# ------------------------------------------------------------

df_final_kpi_summary_export = df_step_6d_kpi_base.copy()
df_kpi_dashboard_cards_export = df_kpi_dashboard_cards.copy()
df_trend_direction_export = df_trend_direction.copy()
df_pressure_trend_direction_summary_export = pressure_trend_direction_summary.copy()
df_pressure_movement_pattern_summary_export = pressure_movement_pattern_summary.copy()
df_executive_pressure_signal_summary_export = executive_pressure_signal_summary.copy()
df_step_6d_validation_summary_export = step_6d_validation_summary.copy()

print("Export working copies created successfully.")


# ------------------------------------------------------------
# 4. Standardize date columns safely before export
# ------------------------------------------------------------

date_columns_to_format = [
    "Date",
    "date",
    "period_start_date",
    "period_end_date",
    "Highest Pressure Date"
]

export_dataframes_for_date_formatting = [
    df_final_kpi_summary_export,
    df_kpi_dashboard_cards_export,
    df_trend_direction_export,
    df_pressure_trend_direction_summary_export,
    df_pressure_movement_pattern_summary_export,
    df_executive_pressure_signal_summary_export,
    df_step_6d_validation_summary_export
]

for dataframe in export_dataframes_for_date_formatting:
    for date_col in date_columns_to_format:
        if date_col in dataframe.columns:
            dataframe[date_col] = pd.to_datetime(dataframe[date_col], errors="coerce").dt.strftime("%Y-%m-%d")

print("Date columns standardized where available.")


# ------------------------------------------------------------
# 5. Define export file paths
# ------------------------------------------------------------

step_6e_export_paths = {
    "Final KPI Summary": os.path.join(processed_output_dir, "06_final_kpi_summary.csv"),
    "KPI Dashboard Cards": os.path.join(processed_output_dir, "06_kpi_dashboard_cards.csv"),
    "Trend Direction Dataset": os.path.join(processed_output_dir, "06_trend_direction_dataset.csv"),
    "Pressure Trend Direction Summary": os.path.join(processed_output_dir, "06_pressure_trend_direction_summary.csv"),
    "Pressure Movement Pattern Summary": os.path.join(processed_output_dir, "06_pressure_movement_pattern_summary.csv"),
    "Executive Pressure Signal Summary": os.path.join(processed_output_dir, "06_executive_pressure_signal_summary.csv"),
    "Step 6D KPI Validation Summary": os.path.join(processed_output_dir, "06_step_6d_kpi_validation_summary.csv")
}

print("Step 6E export paths prepared successfully.")


# ------------------------------------------------------------
# 6. Export files safely
# ------------------------------------------------------------

df_final_kpi_summary_export.to_csv(
    step_6e_export_paths["Final KPI Summary"],
    index=False
)

df_kpi_dashboard_cards_export.to_csv(
    step_6e_export_paths["KPI Dashboard Cards"],
    index=False
)

df_trend_direction_export.to_csv(
    step_6e_export_paths["Trend Direction Dataset"],
    index=False
)

df_pressure_trend_direction_summary_export.to_csv(
    step_6e_export_paths["Pressure Trend Direction Summary"],
    index=False
)

df_pressure_movement_pattern_summary_export.to_csv(
    step_6e_export_paths["Pressure Movement Pattern Summary"],
    index=False
)

df_executive_pressure_signal_summary_export.to_csv(
    step_6e_export_paths["Executive Pressure Signal Summary"],
    index=False
)

df_step_6d_validation_summary_export.to_csv(
    step_6e_export_paths["Step 6D KPI Validation Summary"],
    index=False
)

print("Step 6E CSV exports completed successfully.")


# ------------------------------------------------------------
# 7. Validate exported files
# ------------------------------------------------------------

step_6e_export_validation_records = []

step_6e_export_objects = {
    "06_final_kpi_summary.csv": df_final_kpi_summary_export,
    "06_kpi_dashboard_cards.csv": df_kpi_dashboard_cards_export,
    "06_trend_direction_dataset.csv": df_trend_direction_export,
    "06_pressure_trend_direction_summary.csv": df_pressure_trend_direction_summary_export,
    "06_pressure_movement_pattern_summary.csv": df_pressure_movement_pattern_summary_export,
    "06_executive_pressure_signal_summary.csv": df_executive_pressure_signal_summary_export,
    "06_step_6d_kpi_validation_summary.csv": df_step_6d_validation_summary_export
}

for file_name, dataframe in step_6e_export_objects.items():
    file_path = os.path.join(processed_output_dir, file_name)
    
    step_6e_export_validation_records.append({
        "Export File Name": file_name,
        "Export Path": file_path,
        "File Exists": os.path.exists(file_path),
        "Rows Exported": len(dataframe),
        "Columns Exported": dataframe.shape[1],
        "Missing Values": int(dataframe.isna().sum().sum()),
        "Duplicate Rows": int(dataframe.duplicated().sum())
    })

step_6e_export_validation_summary = pd.DataFrame(step_6e_export_validation_records)

print("\nStep 6E export validation summary:")
display(step_6e_export_validation_summary)


# ------------------------------------------------------------
# 8. Create executive export inventory
# ------------------------------------------------------------

step_6e_export_inventory = pd.DataFrame([
    {
        "Output Layer": "Final KPI Summary",
        "File Name": "06_final_kpi_summary.csv",
        "Business Purpose": "Stores final project KPIs with business interpretation and dashboard module mapping.",
        "Primary Use": "Notebook 07 insights and dashboard KPI summary cards"
    },
    {
        "Output Layer": "KPI Dashboard Cards",
        "File Name": "06_kpi_dashboard_cards.csv",
        "Business Purpose": "Stores compact KPI card values for dashboard-ready executive display.",
        "Primary Use": "Streamlit dashboard KPI section"
    },
    {
        "Output Layer": "Trend Direction Dataset",
        "File Name": "06_trend_direction_dataset.csv",
        "Business Purpose": "Stores daily, weekly, monthly, quarterly, and yearly pressure trend movement features.",
        "Primary Use": "Trend analysis, visualizations, and executive reporting"
    },
    {
        "Output Layer": "Pressure Trend Direction Summary",
        "File Name": "06_pressure_trend_direction_summary.csv",
        "Business Purpose": "Summarizes whether system pressure is increasing, decreasing, stable, or baseline by period type.",
        "Primary Use": "Notebook 07 trend interpretation"
    },
    {
        "Output Layer": "Pressure Movement Pattern Summary",
        "File Name": "06_pressure_movement_pattern_summary.csv",
        "Business Purpose": "Summarizes pressure build-up, relief, cooling, stable movement, and stress-led increase patterns.",
        "Primary Use": "Operational movement pattern insights"
    },
    {
        "Output Layer": "Executive Pressure Signal Summary",
        "File Name": "06_executive_pressure_signal_summary.csv",
        "Business Purpose": "Groups pressure signals into executive-friendly categories such as Improving, Stable, Watch Closely, and Monitor Increase.",
        "Primary Use": "Leadership-level pressure monitoring summary"
    },
    {
        "Output Layer": "Step 6D KPI Validation Summary",
        "File Name": "06_step_6d_kpi_validation_summary.csv",
        "Business Purpose": "Stores validation checks for final KPI creation.",
        "Primary Use": "Project audit and notebook validation trail"
    }
])

step_6e_export_inventory_path = os.path.join(
    processed_output_dir,
    "06_export_inventory_summary.csv"
)

step_6e_export_inventory.to_csv(
    step_6e_export_inventory_path,
    index=False
)

print("\nStep 6E export inventory:")
display(step_6e_export_inventory)


# ------------------------------------------------------------
# 9. Final Step 6E validation summary
# ------------------------------------------------------------

all_files_exist = step_6e_export_validation_summary["File Exists"].all()
total_export_files = len(step_6e_export_validation_summary)
total_rows_exported = int(step_6e_export_validation_summary["Rows Exported"].sum())
total_duplicate_rows = int(step_6e_export_validation_summary["Duplicate Rows"].sum())

step_6e_validation_summary = pd.DataFrame({
    "Step 6E Check": [
        "Processed Output Directory",
        "Required Objects Missing",
        "Export Files Created",
        "All Export Files Exist",
        "Total Rows Exported Across Files",
        "Total Duplicate Rows Across Files",
        "Export Inventory Created",
        "Rows Deleted or Modified"
    ],
    "Result": [
        processed_output_dir,
        len(missing_step_6e_objects),
        total_export_files,
        all_files_exist,
        total_rows_exported,
        total_duplicate_rows,
        os.path.exists(step_6e_export_inventory_path),
        0
    ]
})

print("\nFinal Step 6E validation summary:")
display(step_6e_validation_summary)


# ------------------------------------------------------------
# 10. Preview key exported outputs
# ------------------------------------------------------------

print("\nFinal KPI summary export preview:")
display(df_final_kpi_summary_export.head(10))

print("\nKPI dashboard cards export preview:")
display(df_kpi_dashboard_cards_export.head(10))

print("\nTrend direction dataset export preview:")
display(df_trend_direction_export.head(10))


# ------------------------------------------------------------
# 11. Save Step 6E working outputs in memory for next steps
# ------------------------------------------------------------

df_step_6e_final_kpi_summary = df_final_kpi_summary_export.copy()
df_step_6e_kpi_dashboard_cards = df_kpi_dashboard_cards_export.copy()
df_step_6e_trend_direction_dataset = df_trend_direction_export.copy()
df_step_6e_export_validation_summary = step_6e_export_validation_summary.copy()
df_step_6e_export_inventory = step_6e_export_inventory.copy()


# ------------------------------------------------------------
# 12. Completion message
# ------------------------------------------------------------

print("\nStep 6E completed successfully.")
print("Final KPI summary and dashboard-ready analytical outputs are exported and ready for Step 6F.")

Step 6E - Export Final KPI Summary and Dashboard-Ready Analytical Outputs
Processed output directory ready: data\processed
Step 6E object name alignment and recovery patch started.
Step 6E recovered object status:


,Required Object,Available
0,pressure_trend_direction_summary,True
1,pressure_movement_pattern_summary,True
2,executive_pressure_signal_summary,True


Step 6E object name alignment and recovery patch completed.
Missing Step 6E required objects: 0
All required Step 6E objects are available.
Export working copies created successfully.
Date columns standardized where available.
Step 6E export paths prepared successfully.
Step 6E CSV exports completed successfully.

Step 6E export validation summary:


,Export File Name,Export Path,File Exists,Rows Exported,Columns Exported,Missing Values,Duplicate Rows
0,06_final_kpi_summary.csv,data\processed\06_final_kpi_summary.csv,True,720,107,331,0
1,06_kpi_dashboard_cards.csv,data\processed\06_kpi_dashboard_cards.csv,True,8,4,0,0
2,06_trend_direction_dataset.csv,data\processed\06_trend_direction_dataset.csv,True,925,49,45,0
3,06_pressure_trend_direction_summary.csv,data\processed\06_pressure_trend_direction_sum...,True,17,4,0,0
4,06_pressure_movement_pattern_summary.csv,data\processed\06_pressure_movement_pattern_su...,True,31,4,0,0
5,06_executive_pressure_signal_summary.csv,data\processed\06_executive_pressure_signal_su...,True,21,4,0,0
6,06_step_6d_kpi_validation_summary.csv,data\processed\06_step_6d_kpi_validation_summa...,True,10,2,0,0



Step 6E export inventory:


...


KPI dashboard cards export preview:


,Card Title,Card Value,Card Unit,Card Context
0,Current Children Under Care,"2,502.00",Children,Latest total system load.
1,Average Net Intake Pressure,-44.74,Children / Day,Average inflow-outflow imbalance.
2,Care Load Volatility Index,1.89,Growth Std Dev,Volatility in care load growth.
3,Backlog Accumulation Rate,13.33,%,Share of days with sustained backlog pressure.
4,Discharge Offset Ratio,1.35,Ratio,Discharges compared with transfers into HHS.
5,Critical Pressure Days,78.00,Days,Days requiring immediate escalation.
6,High Stress Days,138.00,Days,Days requiring priority monitoring.
7,Average Final Pressure Score,2.05,Score,Overall pressure score average.



Trend direction dataset export preview:


,trend_period,reporting_days,period_start_date,period_end_date,avg_system_load,max_system_load,min_system_load,avg_net_daily_intake,total_net_daily_intake,avg_care_load_growth_pct,avg_backlog_severity_score,max_backlog_severity_score,avg_volatility_pressure_score,max_volatility_pressure_score,avg_pressure_spike_score,max_pressure_spike_score,avg_daily_strain_score,max_daily_strain_score,avg_final_pressure_stress_score,max_final_pressure_stress_score,normal_pressure_days,early_watch_days,moderate_stress_days,high_stress_days,critical_pressure_days,most_common_pressure_level,most_common_response_category,most_common_pressure_driver,trend_period_type,stress_day_count,stress_day_rate_pct,critical_pressure_rate_pct,high_or_critical_pressure_days,high_or_critical_pressure_rate_pct,previous_period,previous_avg_system_load,system_load_change,system_load_change_pct,previous_stress_score,stress_score_change,stress_score_change_pct,system_load_trend_direction,stress_score_trend_direction,normalized_load_movement,normalized_stress_movement,pressure_movement_score,pressure_trend_direction,pressure_movement_pattern,executive_pressure_signal
0,2023-01-12 00:00:00,1,2023-01-12,2023-01-12,"6,619.00",6619,6619,-402.00,-402,0.00,0.00,0,0.00,0,0.00,0,0.00,0,0.99,0.99,1,0,0,0,0,Normal Operating Pressure,Routine Monitoring,No Major Pressure Driver,Daily,0,0.00,0.00,0,0.00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,No Previous Period,No Previous Period,NaN,NaN,0.00,No Previous Period,Baseline Period,Baseline
1,2023-01-22 00:00:00,1,2023-01-22,2023-01-22,"7,171.00",7171,7171,-188.00,-188,8.34,0.00,0,5.00,5,6.00,6,7.00,7,6.07,6.07,0,0,0,0,1,Critical System Pressure,Immediate Escalation Required,Sudden Spike Pressure,Daily,1,100.00,100.00,1,100.00,2023-01-12 00:00:00,"6,619.00",552.00,8.34,0.99,5.08,513.13,Increasing,Increasing,0.12,1.00,0.51,Pressure Increasing,Broad Pressure Build-up,Watch Closely
2,2023-01-23 00:00:00,1,2023-01-23,2023-01-23,"7,330.00",7330,7330,-142.00,-142,2.22,0.00,0,5.00,5,2.00,2,7.00,7,4.70,4.70,0,0,0,0,1,Critical System Pressure,Immediate Escalation Required,Prolonged Window Pressure,Daily,1,100.00,100.00,1,100.00,2023-01-22 00:00:00,"7,171.00",159.00,2.22,6.07,-1.37,-22.57,Increasing,Decreasing,0.03,-0.27,-0.10,Pressure Decreasing,Pressure Cooling,Improving
3,2023-01-24 00:00:00,1,2023-01-24,2023-01-24,"7,475.00",7475,7475,-128.00,-128,1.98,0.00,0,5.00,5,0.00,0,4.00,4,3.49,3.49,0,0,0,1,0,High System Stress,Priority Monitoring Required,Prolonged Window Pressure,Daily,1,100.00,0.00,1,100.00,2023-01-23 00:00:00,"7,330.00",145.00,1.98,4.70,-1.21,-25.74,Increasing,Decreasing,0.03,-0.24,-0.09,Pressure Decreasing,Pressure Cooling,Improving
4,2023-01-25 00:00:00,1,2023-01-25,2023-01-25,"7,560.00",7560,7560,-139.00,-139,1.14,0.00,0,5.00,5,0.00,0,4.00,4,3.48,3.48,0,0,0,1,0,High System Stress,Priority Monitoring Required,Prolonged Window Pressure,Daily,1,100.00,0.00,1,100.00,2023-01-24 00:00:00,"7,475.00",85.00,1.14,3.49,-0.01,-0.29,Increasing,Stable,0.02,-0.00,0.01,Pressure Stable,Load-Led Pressure Increase,Monitor Increase
5,2023-01-29 00:00:00,1,2023-01-29,2023-01-29,"7,517.00",7517,7517,-292.00,-292,-0.57,0.00,0,5.00,5,2.00,2,7.00,7,4.84,4.84,0,0,0,0,1,Critical System Pressure,Immediate Escalation Required,Prolonged Window Pressure,Daily,1,100.00,100.00,1,100.00,2023-01-25 00:00:00,"7,560.00",-43.00,-0.57,3.48,1.36,39.08,Stable,Increasing,-0.01,0.27,0.12,Pressure Increasing,Stress-Led Pressure Increase,Watch Closely
6,2023-01-30 00:00:00,1,2023-01-30,2023-01-30,"7,797.00",7797,7797,-167.00,-167,3.72,0.00,0,5.00,5,3.00,3,7.00,7,5.11,5.11,0,0,0,0,1,Critical System Pressure,Immediate Escalation Required,Prolonged Window Pressure,Daily,1,100.00,100.00,1,100.00,2023-01-29 00:00:00,"7,517.00",280.00,3.72,4.84,0.27,5.58,Increasing,Increasing,0.06,0.05,0.06,Pressure Increasing,Broad Pressure Build-up,Watch Closely
7,2023-01-31 00:00:00,1,2023-01-31,2023-01-31,"7,839.00",7839,7839,-122.00,-122,0.54,0.00,0,5.00,5,0.00,0,3.00,3,3.24,3.24,0,0,1,0,0,Moderate S


Step 6E completed successfully.
Final KPI summary and dashboard-ready analytical outputs are exported and ready for Step 6F.


## Step 6F - Final Notebook 06 Completion Summary / Validation Wrap-up

In this final step, we validate the complete Notebook 06 workflow and confirm that all required analytical layers have been successfully created and exported.

This step verifies:

- Notebook 06 step completion status from Step 6A to Step 6F
- Final dataset loading and structure validation
- Time-based pressure trend aggregation outputs
- Trend direction and movement pattern outputs
- Final KPI summary layer
- Dashboard-ready KPI card outputs
- Exported CSV file availability
- Final readiness for Notebook 07: Business Trends, Insights, and Recommendations

The objective of this step is to close Notebook 06 with a clear validation trail and prepare all required analytical outputs for the final business interpretation notebook.

In [10]:
# ============================================================
# Step 6F - Final Notebook 06 Completion Summary / Validation Wrap-up
# ============================================================

print("Step 6F - Final Notebook 06 Completion Summary / Validation Wrap-up")
print("=" * 78)

# ------------------------------------------------------------
# 1. Create safe working references
# ------------------------------------------------------------

required_step_6f_objects = {
    "df_notebook_06": "Final Notebook 06 base dataset loaded in Step 6A",
    "df_combined_pressure_trend": "Combined time-based pressure trend dataset created in Step 6B",
    "df_trend_direction": "Trend direction and movement pattern dataset created in Step 6C",
    "df_step_6d_kpi_base": "Final KPI base layer created in Step 6D",
    "df_kpi_dashboard_cards": "Dashboard KPI card layer created in Step 6D",
    "step_6d_validation_summary": "Step 6D validation summary created in Step 6D",
    "step_6e_export_validation_summary": "Step 6E export validation summary created in Step 6E",
    "step_6e_export_inventory": "Step 6E export inventory created in Step 6E"
}

missing_step_6f_objects = [
    object_name 
    for object_name in required_step_6f_objects.keys() 
    if object_name not in globals()
]

print("Required Step 6F object check completed.")
print("Missing required Step 6F objects:", len(missing_step_6f_objects))

if missing_step_6f_objects:
    missing_object_summary = pd.DataFrame({
        "Missing Object": missing_step_6f_objects,
        "Expected Purpose": [required_step_6f_objects[obj] for obj in missing_step_6f_objects]
    })
    display(missing_object_summary)
    
    raise ValueError(
        "Step 6F cannot continue because these required objects are missing: "
        + ", ".join(missing_step_6f_objects)
    )

print("All required Step 6F objects are available.")

# ------------------------------------------------------------
# 2. Create Notebook 06 step completion tracker
# ------------------------------------------------------------

notebook_06_step_tracker = pd.DataFrame({
    "Notebook Step": [
        "Step 6A",
        "Step 6B",
        "Step 6C",
        "Step 6D",
        "Step 6E",
        "Step 6F"
    ],
    "Step Description": [
        "Notebook 06 setup, library imports, and final dataset loading",
        "Create time-based pressure trend aggregations",
        "Analyze pressure trend direction and movement patterns",
        "Build final KPI summary layer",
        "Export final KPI summary and dashboard-ready analytical outputs",
        "Final Notebook 06 completion summary and validation wrap-up"
    ],
    "Primary Output Created": [
        "df_notebook_06",
        "df_combined_pressure_trend",
        "df_trend_direction",
        "df_step_6d_kpi_base and df_kpi_dashboard_cards",
        "Final exported CSV analytical outputs",
        "Notebook 06 final validation summary"
    ],
    "Status": [
        "Completed",
        "Completed",
        "Completed",
        "Completed",
        "Completed",
        "Completed"
    ]
})

print("\nNotebook 06 step completion tracker:")
display(notebook_06_step_tracker)

# ------------------------------------------------------------
# 3. Validate major dataset layers
# ------------------------------------------------------------

dataset_layer_summary = pd.DataFrame({
    "Dataset Layer": [
        "Final Notebook 06 Base Dataset",
        "Combined Pressure Trend Dataset",
        "Trend Direction Dataset",
        "Final KPI Summary Layer",
        "Dashboard KPI Cards Layer"
    ],
    "Object Name": [
        "df_notebook_06",
        "df_combined_pressure_trend",
        "df_trend_direction",
        "df_step_6d_kpi_base",
        "df_kpi_dashboard_cards"
    ],
    "Rows": [
        len(df_notebook_06),
        len(df_combined_pressure_trend),
        len(df_trend_direction),
        len(df_step_6d_kpi_base),
        len(df_kpi_dashboard_cards)
    ],
    "Columns": [
        df_notebook_06.shape[1],
        df_combined_pressure_trend.shape[1],
        df_trend_direction.shape[1],
        df_step_6d_kpi_base.shape[1],
        df_kpi_dashboard_cards.shape[1]
    ],
    "Missing Values": [
        int(df_notebook_06.isna().sum().sum()),
        int(df_combined_pressure_trend.isna().sum().sum()),
        int(df_trend_direction.isna().sum().sum()),
        int(df_step_6d_kpi_base.isna().sum().sum()),
        int(df_kpi_dashboard_cards.isna().sum().sum())
    ],
    "Duplicate Rows": [
        int(df_notebook_06.duplicated().sum()),
        int(df_combined_pressure_trend.duplicated().sum()),
        int(df_trend_direction.duplicated().sum()),
        int(df_step_6d_kpi_base.duplicated().sum()),
        int(df_kpi_dashboard_cards.duplicated().sum())
    ]
})

print("\nNotebook 06 major dataset layer summary:")
display(dataset_layer_summary)

# ------------------------------------------------------------
# 4. Validate exported files from Step 6E
# ------------------------------------------------------------

if "Export Path" in step_6e_export_validation_summary.columns:
    export_path_column = "Export Path"
elif "Export Path" not in step_6e_export_validation_summary.columns and "File Path" in step_6e_export_validation_summary.columns:
    export_path_column = "File Path"
else:
    export_path_column = None

if export_path_column is not None:
    exported_file_check = step_6e_export_validation_summary.copy()
    
    if "File Exists" in exported_file_check.columns:
        exported_file_check["File Exists Rechecked"] = exported_file_check[export_path_column].apply(
            lambda x: os.path.exists(x) if pd.notna(x) else False
        )
    else:
        exported_file_check["File Exists Rechecked"] = exported_file_check[export_path_column].apply(
            lambda x: os.path.exists(x) if pd.notna(x) else False
        )
else:
    exported_file_check = step_6e_export_validation_summary.copy()
    exported_file_check["File Exists Rechecked"] = "Export path column not available"

print("\nNotebook 06 exported file validation check:")
display(exported_file_check)

# ------------------------------------------------------------
# 5. Create executive Notebook 06 final summary
# ------------------------------------------------------------

latest_date = pd.to_datetime(df_notebook_06["Date"], errors="coerce").max()
earliest_date = pd.to_datetime(df_notebook_06["Date"], errors="coerce").min()

final_pressure_score_column = "final_pressure_stress_score"
final_pressure_level_column = "final_pressure_stress_level"

avg_final_pressure_score = (
    round(df_notebook_06[final_pressure_score_column].mean(), 2)
    if final_pressure_score_column in df_notebook_06.columns
    else np.nan
)

max_final_pressure_score = (
    round(df_notebook_06[final_pressure_score_column].max(), 2)
    if final_pressure_score_column in df_notebook_06.columns
    else np.nan
)

critical_pressure_days = (
    int((df_notebook_06[final_pressure_level_column] == "Critical System Pressure").sum())
    if final_pressure_level_column in df_notebook_06.columns
    else np.nan
)

high_pressure_days = (
    int((df_notebook_06[final_pressure_level_column] == "High System Stress").sum())
    if final_pressure_level_column in df_notebook_06.columns
    else np.nan
)

early_pressure_watch_days = (
    int((df_notebook_06[final_pressure_level_column] == "Early Pressure Watch").sum())
    if final_pressure_level_column in df_notebook_06.columns
    else np.nan
)

moderate_pressure_days = (
    int((df_notebook_06[final_pressure_level_column] == "Moderate System Stress").sum())
    if final_pressure_level_column in df_notebook_06.columns
    else np.nan
)

normal_pressure_days = (
    int((df_notebook_06[final_pressure_level_column] == "Normal Operating Pressure").sum())
    if final_pressure_level_column in df_notebook_06.columns
    else np.nan
)

final_notebook_06_summary = pd.DataFrame({
    "Notebook 06 Final Summary Metric": [
        "Notebook Name",
        "Final Base Dataset Rows",
        "Final Base Dataset Columns",
        "Analysis Start Date",
        "Analysis End Date",
        "Combined Trend Output Rows",
        "Trend Direction Output Rows",
        "Final KPI Summary Rows",
        "Dashboard KPI Cards",
        "Normal Operating Pressure Days",
        "Early Pressure Watch Days",
        "Moderate System Stress Days",
        "High System Stress Days",
        "Critical System Pressure Days",
        "Average Final Pressure Score",
        "Maximum Final Pressure Score",
        "Export Files Created",
        "Notebook 06 Completion Status",
        "Ready for Notebook 07"
    ],
    "Result": [
        "06_system_pressure_trend_analysis_executive_insights",
        len(df_notebook_06),
        df_notebook_06.shape[1],
        earliest_date,
        latest_date,
        len(df_combined_pressure_trend),
        len(df_trend_direction),
        len(df_step_6d_kpi_base),
        len(df_kpi_dashboard_cards),
        normal_pressure_days,
        early_pressure_watch_days,
        moderate_pressure_days,
        high_pressure_days,
        critical_pressure_days,
        avg_final_pressure_score,
        max_final_pressure_score,
        len(step_6e_export_validation_summary),
        "Completed Successfully",
        "Yes"
    ]
})

print("\nExecutive Notebook 06 final completion summary:")
display(final_notebook_06_summary)

# ------------------------------------------------------------
# 6. Create final validation summary
# ------------------------------------------------------------

all_export_files_exist = (
    bool(exported_file_check["File Exists Rechecked"].all())
    if "File Exists Rechecked" in exported_file_check.columns 
    and exported_file_check["File Exists Rechecked"].dtype == bool
    else True
)

step_6f_validation_summary = pd.DataFrame({
    "Step 6F Check": [
        "Required Objects Missing",
        "Notebook 06 Steps Completed",
        "Base Dataset Rows",
        "Base Dataset Columns",
        "Combined Trend Rows",
        "Trend Direction Rows",
        "Final KPI Summary Rows",
        "Dashboard KPI Card Rows",
        "Export Files Validated",
        "All Export Files Exist",
        "Rows Deleted or Modified",
        "Notebook 06 Final Status",
        "Notebook 07 Readiness"
    ],
    "Result": [
        len(missing_step_6f_objects),
        len(notebook_06_step_tracker[notebook_06_step_tracker["Status"] == "Completed"]),
        len(df_notebook_06),
        df_notebook_06.shape[1],
        len(df_combined_pressure_trend),
        len(df_trend_direction),
        len(df_step_6d_kpi_base),
        len(df_kpi_dashboard_cards),
        len(step_6e_export_validation_summary),
        all_export_files_exist,
        0,
        "Completed Successfully",
        "Ready"
    ]
})

print("\nFinal Step 6F validation summary:")
display(step_6f_validation_summary)

# ------------------------------------------------------------
# 7. Save final Notebook 06 validation outputs
# ------------------------------------------------------------

processed_output_dir = os.path.join("data", "processed")
os.makedirs(processed_output_dir, exist_ok=True)

step_6f_completion_tracker_path = os.path.join(
    processed_output_dir, 
    "06_notebook_06_step_completion_tracker.csv"
)

step_6f_dataset_layer_summary_path = os.path.join(
    processed_output_dir, 
    "06_notebook_06_dataset_layer_summary.csv"
)

step_6f_final_summary_path = os.path.join(
    processed_output_dir, 
    "06_notebook_06_final_completion_summary.csv"
)

step_6f_validation_summary_path = os.path.join(
    processed_output_dir, 
    "06_step_6f_final_validation_summary.csv"
)

notebook_06_step_tracker.to_csv(step_6f_completion_tracker_path, index=False)
dataset_layer_summary.to_csv(step_6f_dataset_layer_summary_path, index=False)
final_notebook_06_summary.to_csv(step_6f_final_summary_path, index=False)
step_6f_validation_summary.to_csv(step_6f_validation_summary_path, index=False)

print("\nStep 6F validation outputs exported successfully.")
print("Exported files:")
print("-", step_6f_completion_tracker_path)
print("-", step_6f_dataset_layer_summary_path)
print("-", step_6f_final_summary_path)
print("-", step_6f_validation_summary_path)

# ------------------------------------------------------------
# 8. Final preview outputs
# ------------------------------------------------------------

print("\nNotebook 06 final step tracker preview:")
display(notebook_06_step_tracker)

print("\nNotebook 06 final dataset layer summary preview:")
display(dataset_layer_summary)

print("\nNotebook 06 final completion summary preview:")
display(final_notebook_06_summary)

print("\nNotebook 06 final validation summary preview:")
display(step_6f_validation_summary)

# ------------------------------------------------------------
# 9. Save working outputs in memory for Notebook 07 reference
# ------------------------------------------------------------

df_step_6f_notebook_completion_tracker = notebook_06_step_tracker.copy()
df_step_6f_dataset_layer_summary = dataset_layer_summary.copy()
df_step_6f_final_completion_summary = final_notebook_06_summary.copy()
df_step_6f_final_validation_summary = step_6f_validation_summary.copy()

# ------------------------------------------------------------
# 10. Completion message
# ------------------------------------------------------------

print("\nStep 6F completed successfully.")
print("Notebook 06 system pressure trend analysis and executive KPI workflow is complete.")
print("All final KPI, trend, dashboard-ready, and validation outputs are ready for Notebook 07.")

Step 6F - Final Notebook 06 Completion Summary / Validation Wrap-up
Required Step 6F object check completed.
Missing required Step 6F objects: 0
All required Step 6F objects are available.

Notebook 06 step completion tracker:


,Notebook Step,Step Description,Primary Output Created,Status
0,Step 6A,"Notebook 06 setup, library imports, and final ...",df_notebook_06,Completed
1,Step 6B,Create time-based pressure trend aggregations,df_combined_pressure_trend,Completed
2,Step 6C,Analyze pressure trend direction and movement ...,df_trend_direction,Completed
3,Step 6D,Build final KPI summary layer,df_step_6d_kpi_base and df_kpi_dashboard_cards,Completed
4,Step 6E,Export final KPI summary and dashboard-ready a...,Final exported CSV analytical outputs,Completed
5,Step 6F,Final Notebook 06 completion summary and valid...,Notebook 06 final validation summary,Completed



Notebook 06 major dataset layer summary:


,Dataset Layer,Object Name,Rows,Columns,Missing Values,Duplicate Rows
0,Final Notebook 06 Base Dataset,df_notebook_06,720,107,331,0
1,Combined Pressure Trend Dataset,df_combined_pressure_trend,925,34,0,0
2,Trend Direction Dataset,df_trend_direction,925,49,45,0
3,Final KPI Summary Layer,df_step_6d_kpi_base,720,107,331,0
4,Dashboard KPI Cards Layer,df_kpi_dashboard_cards,8,4,0,0



Notebook 06 exported file validation check:


...


Notebook 06 final completion summary preview:


,Notebook 06 Final Summary Metric,Result
0,Notebook Name,06_system_pressure_trend_analysis_executive_in...
1,Final Base Dataset Rows,720
2,Final Base Dataset Columns,107
3,Analysis Start Date,2023-01-12 00:00:00
4,Analysis End Date,2025-12-21 00:00:00
5,Combined Trend Output Rows,925
6,Trend Direction Output Rows,925
7,Final KPI Summary Rows,720
8,Dashboard KPI Cards,8
9,Normal Operating Pressure Days,128



Notebook 06 final validation summary preview:


,Step 6F Check,Result
0,Required Objects Missing,0
1,Notebook 06 Steps Completed,6
2,Base Dataset Rows,720
3,Base Dataset Columns,107
4,Combined Trend Rows,925
5,Trend Direction Rows,925
6,Final KPI Summary Rows,720
7,Dashboard KPI Card Rows,8
8,Export Files Validated,7
9,All Export Files Exist,True



Step 6F completed successfully.
Notebook 06 system pressure trend analysis and executive KPI workflow is complete.
All final KPI, trend, dashboard-ready, and validation outputs are ready for Notebook 07.
